In [123]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
import glob
import tqdm
plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 2})

In [124]:
label_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/label/Baeklab.070.GP3.depth5_None.twm6astrict.tsv", sep="\t")
label_df = label_df[["id", "depth", "label", "m6A_level", "5mer", "drach"]]
label_df.rename(columns = {"id": "label_id","m6A_level": "dom_label"}, inplace = True)
label_df_filtered = label_df[((label_df["dom_label"] > 0.0) & (label_df["label"] == 1)) | (label_df["label"] == 0)]
print(label_df_filtered["label"].value_counts())

label
0    18399879
1      100118
Name: count, dtype: int64


In [125]:
logsum_df = "/extdata4/baeklab/Hyeonseo/m6A/inference/inference/AIRNA-DW-v4-20240905-092858-10-177000-token_normalise_dwell_bq_npz_allmotif_pileup_psum/pileup.npz"
with np.load(logsum_df, allow_pickle=True) as data:
    logsum_df = pd.DataFrame({key: data[key] for key in data.keys()})
logsum_df = logsum_df[logsum_df["count_dom"] >= 20]
logsum_df = logsum_df.merge(label_df, on="label_id", how="inner")
print(logsum_df)

                label_id      p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
0         NM_000016:1000   0.606511      0.000000        0.000000          0   
1         NM_000016:1008   0.948786      0.000000        0.000000          0   
2         NM_000016:1009   2.607620      0.000000        0.000000          0   
3         NM_000016:1010   4.912612      0.000000        0.000000          0   
4         NM_000016:1014   3.237954      0.000000        0.000000          0   
...                  ...        ...           ...             ...        ...   
15992331    NR_184306:85   1.133184      0.000000        0.000000          0   
15992332    NR_184306:87   3.771550     -0.001452       -2.476578          1   
15992333    NR_184306:88   2.137289      0.000000        0.000000          0   
15992334    NR_184306:93  32.341377     -0.011076      -78.301079         23   
15992335    NR_184306:98   4.046034      0.000000        0.000000          0   

          logsum_p_neg  logsum_1_p_neg 

In [126]:
dorado_080_df = pd.read_pickle("/extdata4/baeklab/Hyeonseo/m6A/dorado-080/ON0090_dorado_basecall.bed" + ".m6A.pkl")
dorado_080_df.rename(columns = {"count_dom": "dorado_count", "dom": "pred_dorado"}, inplace = True)
dorado_080_df = dorado_080_df[dorado_080_df["dorado_count"] >= 20]
dorado_080_df = dorado_080_df.merge(label_df, on="label_id", how="inner")
dorado_080_df = dorado_080_df[["pred_dorado", "dorado_count", "label_id"]]
print(dorado_080_df)    

          pred_dorado  dorado_count       label_id
0              0.0161            62    NM_000016:2
1              0.0149            67    NM_000016:5
2              0.0125           160   NM_000016:21
3              0.0616           146   NM_000016:24
4              0.0412           170   NM_000016:29
...               ...           ...            ...
15312228       0.1463            41  NR_184306:326
15312229       0.0000            40  NR_184306:330
15312230       0.0233            43  NR_184306:331
15312231       0.0000            43  NR_184306:333
15312232       0.0938            32  NR_184306:334

[15312233 rows x 3 columns]


In [127]:
dorado_080_df.rename(columns = {"dorado_count": "dorado_count_080", "pred_dorado": "pred_dorado_080"}, inplace = True)
eval_df = logsum_df.merge(dorado_080_df, on="label_id", how="left").fillna(0)
print(eval_df)

                label_id      p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
0         NM_000016:1000   0.606511      0.000000        0.000000          0   
1         NM_000016:1008   0.948786      0.000000        0.000000          0   
2         NM_000016:1009   2.607620      0.000000        0.000000          0   
3         NM_000016:1010   4.912612      0.000000        0.000000          0   
4         NM_000016:1014   3.237954      0.000000        0.000000          0   
...                  ...        ...           ...             ...        ...   
15992369    NR_184306:85   1.133184      0.000000        0.000000          0   
15992370    NR_184306:87   3.771550     -0.001452       -2.476578          1   
15992371    NR_184306:88   2.137289      0.000000        0.000000          0   
15992372    NR_184306:93  32.341377     -0.011076      -78.301079         23   
15992373    NR_184306:98   4.046034      0.000000        0.000000          0   

          logsum_p_neg  logsum_1_p_neg 

In [128]:
eval_df["pm6a"] =  -(2-eval_df["dom"])*eval_df["logsum_1_p_pos"]/eval_df["count_all"] + ((1-eval_df["dom"])*np.log10(np.clip(1-eval_df["dom"],1e-30,1)) + eval_df["dom"] * np.log10(np.clip(eval_df["dom"],1e-30,1)))*(eval_df["count_pos"]/eval_df["count_all"])

print(eval_df[['pm6a', 'dom', 'count_dom', 'count_all', '5mer', 'drach', 'pred_dorado_080', 'dorado_count_080']])

              pm6a       dom  count_dom  count_all   5mer  drach  \
0         0.000000  0.000000        258        258  AAACT   True   
1         0.000000  0.000000        256        257  GGAAA  False   
2        -0.000000  0.003891        257        260  GAAAG  False   
3        -0.000000  0.011583        259        263  AAAGC  False   
4         0.000000  0.000000        252        258  CTACT  False   
...            ...       ...        ...        ...    ...    ...   
15992369  0.000000  0.000000         49         51  TTATA  False   
15992370  0.092186  0.061224         49         51  ATAAT  False   
15992371 -0.000000  0.039216         51         51  TAATC  False   
15992372  1.919874  0.695652         46         50  GGACT   True   
15992373  0.000000  0.000000         40         48  TCAGC  False   

          pred_dorado_080  dorado_count_080  
0                  0.0076             263.0  
1                  0.0000             262.0  
2                  0.0039             257.0  

In [129]:

geneid_table = pd.read_pickle("/extdata4/baeklab/Hyeonseo/m6A/res/ref/GRCh38_latest_genomic.convert_table.pkl")
geneid_table.rename({"transcript_id":"refseq"}, axis=1, inplace=True)
print(geneid_table)

              gene_id        refseq
6             DDX11L1     NR_046018
11             WASH7P     NR_024540
24          MIR6859-1     NR_106918
31        MIR1302-2HG  XR_007065314
36          MIR1302-2     NR_036051
...               ...           ...
4683855     KIR2DS5_6  XM_054333504
4683875    KIR2DL5A_7  XM_054333508
4683946     KIR3DS1_9  XM_054333507
4683987  LOC128966733  XR_008485847
4683991  LOC128966733  XM_054333509

[184489 rows x 2 columns]


In [130]:
eval_df["refseq"] = eval_df["label_id"].str.split(":").str[0]
eval_df = eval_df.merge(geneid_table, on="refseq", how="left")
eval_df.rename(columns = {"pm6a": "log_pm6a", "pred_dorado_080": "pred_dorado", "dorado_count_080": "count_dorado"}, inplace=True)
print(eval_df)

                label_id      p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
0         NM_000016:1000   0.606511      0.000000        0.000000          0   
1         NM_000016:1008   0.948786      0.000000        0.000000          0   
2         NM_000016:1009   2.607620      0.000000        0.000000          0   
3         NM_000016:1010   4.912612      0.000000        0.000000          0   
4         NM_000016:1014   3.237954      0.000000        0.000000          0   
...                  ...        ...           ...             ...        ...   
15992369    NR_184306:85   1.133184      0.000000        0.000000          0   
15992370    NR_184306:87   3.771550     -0.001452       -2.476578          1   
15992371    NR_184306:88   2.137289      0.000000        0.000000          0   
15992372    NR_184306:93  32.341377     -0.011076      -78.301079         23   
15992373    NR_184306:98   4.046034      0.000000        0.000000          0   

          logsum_p_neg  logsum_1_p_neg 

In [131]:
ivt_tpm_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0105/ON0105/result/tpm/tpm.tsv",sep="\t")
ivt_tpm_df = ivt_tpm_df[ivt_tpm_df["tpm"] > 0.0]
ivt_tpm_df["logtpm"] = np.log10(ivt_tpm_df["tpm"])
ivt_tpm_df["refseq"] = ivt_tpm_df["transcript_name"].str.split(".").str[0]
ivt_tpm_df["ivt_perc"] = ivt_tpm_df["logtpm"].rank(pct=True)
ivt_tpm_df

,transcript_name,raw,est_count,tpm,transcript_length,logtpm,refseq,ivt_perc
0,NM_002952.4,1.339165e-02,1.840900e+04,1.339165e+04,945,4.126834,NM_002952,1.000000
1,NR_026712.1,1.118017e-02,1.536897e+04,1.118017e+04,658,4.048449,NR_026712,0.999981
2,NM_001000.4,9.515059e-03,1.308000e+04,9.515059e+03,390,3.978411,NM_001000,0.999961
3,NM_022551.3,9.240792e-03,1.270297e+04,9.240792e+03,549,3.965709,NM_022551,0.999942
4,NM_001404.5,9.131693e-03,1.255300e+04,9.131693e+03,1446,3.960551,NM_001404,0.999922
...,...,...,...,...,...,...,...,...
51536,NR_024240.1,3.105177e-40,4.268573e-34,3.105177e-34,1580,-33.507914,NR_024240,0.000097
51537,NR_024219.1,6.239105e-49,8.576667e-43,6.239105e-43,120,-42.204878,NR_024219,0.000058
51538,NR_024217.1,6.239105e-49,8.576667e-43,6.239105e-43,120,-42.204878,NR_024217,0.000058
51539,NR_024220.1,6.239105e-49,8.576667e-43,6.239105e-43,120,-42.204878,NR_024220,0.000058


In [132]:
cell_tpm_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/result/dorado-070-aligned/tpm/tpm.tsv",sep="\t")
cell_tpm_df = cell_tpm_df[cell_tpm_df["tpm"] > 0.0]
cell_tpm_df["logtpm"] = np.log10(cell_tpm_df["tpm"])
cell_tpm_df["refseq"] = cell_tpm_df["transcript_name"].str.split(".").str[0]
cell_tpm_df["cell_perc"] = cell_tpm_df["logtpm"].rank(pct=True)
cell_tpm_df

,transcript_name,raw,est_count,tpm,transcript_length,logtpm,refseq,cell_perc
0,NM_002952.4,1.124904e-02,4.911100e+04,1.124904e+04,945,4.051116,NM_002952,1.000000
1,NM_022551.3,6.785937e-03,2.962600e+04,6.785937e+03,549,3.831610,NM_022551,0.999984
2,NM_001000.4,6.460679e-03,2.820599e+04,6.460679e+03,390,3.810278,NM_001000,0.999967
3,NM_000972.3,5.957222e-03,2.600800e+04,5.957222e+03,887,3.775044,NM_000972,0.999951
4,NR_026712.1,5.942332e-03,2.594299e+04,5.942332e+03,658,3.773957,NR_026712,0.999934
...,...,...,...,...,...,...,...,...
60667,NM_001395980.1,7.291763e-45,3.183433e-38,7.291763e-39,1559,-38.137167,NM_001395980,0.000082
60668,NR_001434.4,2.219039e-46,9.687866e-40,2.219039e-40,1705,-39.653835,NR_001434,0.000066
60669,NR_144546.2,9.444820e-51,4.123413e-44,9.444820e-45,1456,-44.024806,NR_144546,0.000041
60670,NR_176224.1,9.444820e-51,4.123413e-44,9.444820e-45,1414,-44.024806,NR_176224,0.000041


In [133]:
eval_df = eval_df.merge(ivt_tpm_df[["refseq", "ivt_perc"]], on="refseq", how="left")
eval_df = eval_df.merge(cell_tpm_df[["refseq", "cell_perc"]], on="refseq", how="left")
eval_df.fillna(0, inplace=True)
print(eval_df)


                label_id      p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
0         NM_000016:1000   0.606511      0.000000        0.000000          0   
1         NM_000016:1008   0.948786      0.000000        0.000000          0   
2         NM_000016:1009   2.607620      0.000000        0.000000          0   
3         NM_000016:1010   4.912612      0.000000        0.000000          0   
4         NM_000016:1014   3.237954      0.000000        0.000000          0   
...                  ...        ...           ...             ...        ...   
15992369    NR_184306:85   1.133184      0.000000        0.000000          0   
15992370    NR_184306:87   3.771550     -0.001452       -2.476578          1   
15992371    NR_184306:88   2.137289      0.000000        0.000000          0   
15992372    NR_184306:93  32.341377     -0.011076      -78.301079         23   
15992373    NR_184306:98   4.046034      0.000000        0.000000          0   

          logsum_p_neg  logsum_1_p_neg 

In [134]:
eval_df["noisy_or"] = - eval_df["logsum_1_p_pos"] /  eval_df["count_all"]
select_df = eval_df[['label_id', 'gene_id', 'log_pm6a', "noisy_or", 'dom', 'dom_label', 'label', 'count_dom', 'count_all', '5mer', 'drach', 'pred_dorado', 'count_dorado', "ivt_perc", "cell_perc"]].copy()
select_df[select_df["log_pm6a"] > 1.0]

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc
784,NM_000017:1321,ACADS,2.021584,1.425952,0.492063,0.452959,-1,126,131,GGACT,True,0.4634,123.0,0.898421,0.918282
796,NM_000017:1373,ACADS,2.437110,1.898793,0.632000,0.720267,1,125,130,AGACT,True,0.5882,119.0,0.898421,0.918282
2170,NM_000021:1584,PSEN1,3.293502,3.287473,0.992908,0.875758,-1,141,143,GGACC,True,0.9856,139.0,0.825547,0.801762
2191,NM_000021:1655,PSEN1,1.550892,1.055201,0.444444,0.498157,-1,135,145,TGACT,True,0.4191,136.0,0.825547,0.801762
2217,NM_000021:1730,PSEN1,2.817645,2.696066,0.916667,0.916725,1,144,145,GGACT,True,0.9203,138.0,0.825547,0.801762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15990848,NR_184072:1057,ATXN7L3-AS1,1.224421,0.889524,0.517241,0.000000,-3,29,35,GAACT,True,0.0000,0.0,0.137434,0.697447
15990939,NR_184072:1390,ATXN7L3-AS1,1.092904,0.713811,0.375000,0.000000,-3,24,30,GAACT,True,0.0000,0.0,0.137434,0.697447
15990972,NR_184072:502,ATXN7L3-AS1,1.675686,1.527358,0.850000,0.000000,-3,20,25,GGACT,True,0.0000,0.0,0.137434,0.697447
15992368,NR_184306:77,ZNF747-DT,1.229243,0.736066,0.260870,0.000000,-3,46,49,GGACC,True,0.2500,44.0,0.718021,0.683742


In [135]:
select_df[select_df["gene_id"]=="ACTB"]

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc
1257724,NM_001101:1000,ACTB,0.035447,0.018853,0.072576,0.0,0,10086,10918,GTACC,False,0.0762,7050.0,0.998525,0.999506
1257725,NM_001101:1008,ACTB,0.020233,0.010883,0.082912,0.0,-1,9890,11212,GCATT,False,0.0329,6377.0,0.998525,0.999506
1257726,NM_001101:1015,ACTB,0.006398,0.003242,0.013325,0.0,0,11182,11402,CGACA,False,0.0177,8150.0,0.998525,0.999506
1257727,NM_001101:1017,ACTB,0.006861,0.003500,0.019469,0.0,0,10838,11175,ACAGG,False,0.0548,7319.0,0.998525,0.999506
1257728,NM_001101:1020,ACTB,0.000395,0.000198,0.001941,0.0,-1,11336,11377,GGATG,False,0.0116,8269.0,0.998525,0.999506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1258107,NM_001101:972,ACTB,0.006705,0.003473,0.037617,0.0,-1,10607,11300,ACACA,False,0.0564,6893.0,0.998525,0.999506
1258108,NM_001101:974,ACTB,0.006030,0.003073,0.018454,0.0,0,10838,11331,ACAGT,False,0.0375,7042.0,0.998525,0.999506
1258109,NM_001101:990,ACTB,0.009454,0.004902,0.038906,0.0,0,10127,10790,GCACC,False,0.0207,7446.0,0.998525,0.999506
1258110,NM_001101:993,ACTB,0.030987,0.016887,0.097828,0.0,0,9946,11155,CCACC,False,0.0128,7414.0,0.998525,0.999506


In [136]:
select_df["refseq"] = select_df["label_id"].str.split(":").str[0]

In [137]:
m6a_counts = select_df[select_df["log_pm6a"] > np.percentile(select_df["log_pm6a"],99.5)].groupby("refseq").count().sort_values("refseq")["count_all"]
m6a_counts = m6a_counts.reset_index()
m6a_counts = m6a_counts.merge(cell_tpm_df[["refseq", "transcript_length"]], on="refseq", how="left")
m6a_counts.fillna(0)
m6a_counts["m6a_density"] = m6a_counts["count_all"]/m6a_counts["transcript_length"]
m6a_counts.rename(columns = {"count_all": "m6a_count"}, inplace=True)
m6a_counts[m6a_counts["m6a_density"] > 0.005]

,refseq,m6a_count,transcript_length,m6a_density
49,NM_000199,19,2705.0,0.007024
101,NM_000418,22,3624.0,0.006071
144,NM_000676,12,1522.0,0.007884
147,NM_000692,17,3028.0,0.005614
257,NM_001002914,18,2783.0,0.006468
...,...,...,...,...
15284,NR_171561,8,1405.0,0.005694
15285,NR_171563,8,1566.0,0.005109
15291,NR_172519,16,2722.0,0.005878
15323,NR_183329,4,761.0,0.005256


In [138]:
pm6a_cutoff = np.percentile(select_df["log_pm6a"],99.5)

In [144]:
hiconf_df = select_df[(select_df["log_pm6a"] > pm6a_cutoff) & (~select_df["drach"]) & (select_df["cell_perc"] > 0.9)].copy().sort_values("gene_id")
hiconf_df = hiconf_df.merge(m6a_counts, on="refseq", how="left").fillna(0)
hiconf_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density
0,NM_001089:5961,ABCA3,2.685794,2.127319,0.655290,0.711017,1,293,296,GGACG,False,0.6536,280.0,0.952834,0.961819,NM_001089,4,6602.0,0.000606
1,NM_004996:5211,ABCC1,2.338132,1.984957,0.751908,0.709251,-1,262,283,GAACG,False,0.7126,261.0,0.934344,0.964893,NM_004996,14,6504.0,0.002153
2,NM_001198934:1073,ABCC10,2.337938,1.841972,0.647059,0.687953,1,68,75,GGACG,False,0.6579,76.0,0.833143,0.924578,NM_001198934,9,5043.0,0.001785
3,NM_198147:462,ABHD15,2.393320,2.227509,0.872642,0.942997,-1,212,229,GGACG,False,0.8122,197.0,0.810287,0.919411,NM_198147,29,3492.0,0.008305
4,NM_198147:1838,ABHD15,2.998586,2.858266,0.912698,0.950749,-1,252,258,AGACG,False,0.9102,245.0,0.810287,0.919411,NM_198147,29,3492.0,0.008305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,NM_001195605:938,ZNF865,3.145893,2.856116,0.847222,0.897226,-1,72,76,GGACG,False,0.7639,72.0,0.576541,0.906464,NM_001195605,29,3763.0,0.007707
1004,NM_023926:2063,ZSCAN18,2.756144,2.249168,0.701754,0.778869,-1,57,63,GGACG,False,0.7083,48.0,0.817941,0.948312,NM_023926,3,2783.0,0.001078
1005,NM_001145543:1819,ZSCAN18,2.973099,2.354091,0.664835,0.778869,-1,182,189,GGACG,False,0.6532,173.0,0.817941,0.909085,NM_001145543,3,2539.0,0.001182
1006,NM_199341:1013,ZSWIM9,2.064114,1.343236,0.384615,0.596064,-1,52,52,AGACG,False,0.4038,52.0,0.870297,0.905434,NM_199341,11,3600.0,0.003056


In [145]:
hiconf_df = hiconf_df[hiconf_df["m6a_density"] <= 0.005]
hiconf_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density
0,NM_001089:5961,ABCA3,2.685794,2.127319,0.655290,0.711017,1,293,296,GGACG,False,0.6536,280.0,0.952834,0.961819,NM_001089,4,6602.0,0.000606
1,NM_004996:5211,ABCC1,2.338132,1.984957,0.751908,0.709251,-1,262,283,GAACG,False,0.7126,261.0,0.934344,0.964893,NM_004996,14,6504.0,0.002153
2,NM_001198934:1073,ABCC10,2.337938,1.841972,0.647059,0.687953,1,68,75,GGACG,False,0.6579,76.0,0.833143,0.924578,NM_001198934,9,5043.0,0.001785
5,NM_032182:965,ABRAXAS2,1.675232,1.481502,0.789855,0.871889,1,138,150,GTACT,False,0.7883,137.0,0.915679,0.925905,NM_032182,9,2955.0,0.003046
8,NM_005736:1364,ACTR1A,2.895120,2.557538,0.803279,0.825807,1,488,493,GGACG,False,0.7918,490.0,0.977891,0.979900,NM_005736,2,2830.0,0.000707
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,NM_025069:1812,ZNF703,1.689626,1.570630,0.868580,0.918589,1,662,721,ACACT,False,0.7813,567.0,0.963728,0.987853,NM_025069,11,3316.0,0.003317
1004,NM_023926:2063,ZSCAN18,2.756144,2.249168,0.701754,0.778869,-1,57,63,GGACG,False,0.7083,48.0,0.817941,0.948312,NM_023926,3,2783.0,0.001078
1005,NM_001145543:1819,ZSCAN18,2.973099,2.354091,0.664835,0.778869,-1,182,189,GGACG,False,0.6532,173.0,0.817941,0.909085,NM_001145543,3,2539.0,0.001182
1006,NM_199341:1013,ZSWIM9,2.064114,1.343236,0.384615,0.596064,-1,52,52,AGACG,False,0.4038,52.0,0.870297,0.905434,NM_199341,11,3600.0,0.003056


In [180]:
pm6a_cutoff = 0.712
exclude_gene = [
    "MKI67",
    "NTN1",
    "RMI2",
    "MICU1",
    "KAT6A",
    "TNFRSF1A",
    "KHDRBS1",
    "CDV3",
    "PRMT6",
    "KDM4A",
    "CENPB",
    "PXDN",
]
print(pm6a_cutoff)

hiconf_df = select_df[(select_df["log_pm6a"] > pm6a_cutoff) & (~select_df["drach"]) & (select_df["cell_perc"] > 0.9)].copy().sort_values("gene_id")
hiconf_df = hiconf_df.merge(m6a_counts, on="refseq", how="left").fillna(0)

print(len(select_df[select_df["log_pm6a"] > pm6a_cutoff]))

dorado_fail_df = hiconf_df[(hiconf_df["pred_dorado"] == 0)].copy()
dorado_fail_df = dorado_fail_df[~dorado_fail_df["gene_id"].isin(exclude_gene)]


dorado_fail_df

0.712
151687


,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density
108,NM_001013620:638,ALG10B,0.834373,0.634318,0.550000,0.000000,0,20,21,AAAAC,False,0.0,0.0,0.815438,0.931632,NM_001013620,5.0,10050.0,0.000498
114,NM_020690:2983,ANKHD1-EIF4EBP3,0.819586,0.562345,0.416667,0.000000,0,24,25,TGAAC,False,0.0,25.0,0.904775,0.902228,NM_020690,18.0,8328.0,0.002161
165,NM_018171:3043,APPL2,0.712054,0.588520,0.686567,0.000000,0,134,164,CCATC,False,0.0,146.0,0.903446,0.900770,NM_018171,1.0,3171.0,0.000315
274,NM_001377448:4028,BAHCC1,0.932408,0.528186,0.173913,0.335664,-1,23,25,GGACG,False,0.0,0.0,0.773724,0.904577,NM_001377448,11.0,10726.0,0.001026
459,NM_001170629:536,CHD8,1.165228,1.009193,0.746835,0.000000,0,79,84,AGAAC,False,0.0,78.0,0.903009,0.967135,NM_001170629,20.0,8467.0,0.002362
475,NR_171547:2308,CHPF2,2.214009,1.435382,0.380952,0.394277,1,21,21,GGACG,False,0.0,0.0,0.748113,0.921084,NR_171547,13.0,4206.0,0.003091
546,NM_005730:2720,CTDSP2,0.744648,0.654327,0.766055,0.000000,0,218,256,CCATC,False,0.0,246.0,0.935149,0.966443,NM_005730,10.0,4785.0,0.002090
674,NM_080738:2202,EDARADD,0.766543,0.740457,0.901838,0.000000,0,2720,2959,TCAAA,False,0.0,46.0,0.979026,0.927396,NM_080738,6.0,3095.0,0.001939
740,NM_001329090:805,EPHA2,1.608960,1.455806,0.818182,0.941250,-1,22,24,GCACT,False,0.0,0.0,0.872480,0.901347,NM_001329090,10.0,3878.0,0.002579
841,NM_001316938:1630,FBXL12,1.174176,0.702675,0.250000,0.410789,1,20,22,GGACG,False,0.0,0.0,0.942648,0.903827,NM_001316938,6.0,1917.0,0.003130


In [210]:
very_hiconf_df = hiconf_df[(hiconf_df["log_pm6a"] > np.percentile(select_df["log_pm6a"],99.8)) & (hiconf_df["cell_perc"] > 0.90)].copy()
very_hiconf_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density
5,NM_001089:5961,ABCA3,2.685794,2.127319,0.655290,0.711017,1,293,296,GGACG,False,0.6536,280.0,0.952834,0.961819,NM_001089,4.0,6602.0,0.000606
14,NM_198147:462,ABHD15,2.393320,2.227509,0.872642,0.942997,-1,212,229,GGACG,False,0.8122,197.0,0.810287,0.919411,NM_198147,29.0,3492.0,0.008305
16,NM_198147:1838,ABHD15,2.998586,2.858266,0.912698,0.950749,-1,252,258,AGACG,False,0.9102,245.0,0.810287,0.919411,NM_198147,29.0,3492.0,0.008305
27,NM_005736:1364,ACTR1A,2.895120,2.557538,0.803279,0.825807,1,488,493,GGACG,False,0.7918,490.0,0.977891,0.979900,NM_005736,2.0,2830.0,0.000707
32,NM_138422:1052,ADAT3,2.389407,1.779976,0.569767,0.586346,-1,86,93,GGACG,False,0.6180,89.0,0.779622,0.913205,NM_138422,14.0,1599.0,0.008755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2958,NM_003575:2234,ZNF282,3.129509,2.669289,0.761905,0.718320,1,273,276,GGACG,False,0.7547,265.0,0.731175,0.933231,NM_003575,10.0,3651.0,0.002739
2991,NM_001195605:938,ZNF865,3.145893,2.856116,0.847222,0.897226,-1,72,76,GGACG,False,0.7639,72.0,0.576541,0.906464,NM_001195605,29.0,3763.0,0.007707
2993,NM_001195605:1756,ZNF865,3.206811,3.156591,0.965517,0.949771,1,87,90,AGACG,False,0.9512,82.0,0.576541,0.906464,NM_001195605,29.0,3763.0,0.007707
2995,NM_023926:2063,ZSCAN18,2.756144,2.249168,0.701754,0.778869,-1,57,63,GGACG,False,0.7083,48.0,0.817941,0.948312,NM_023926,3.0,2783.0,0.001078


In [139]:
gene_list = very_hiconf_df["gene_id"].unique()

In [141]:
import os, tqdm, xmltodict, requests, glob

def get_pubmed_count(gene):
    API_KEY="e60738641b2884a4933fe8291725f839bd07"
    query_template = '("Nat *"[Journal] OR "Nature"[Journal] OR "Science"[Journal] OR "Cell"[Journal] OR "Cell *"[Journal]) AND "{gene}"[tiab]'
    retmax = 1000
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={term}&retmax={retmax}&usehistory=y&api_key={apikey}"
    response = requests.get(search_url.format(term=query_template.format(gene=gene), retmax=retmax, apikey = API_KEY))
    count = int(xmltodict.parse(response.content)["eSearchResult"]["Count"])
    return count

In [142]:
count_dict = {gene: get_pubmed_count(gene) for gene in tqdm.tqdm(gene_list)}
print(count_dict)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 54/54 [00:43<00:00,  1.24it/s]

{'ABCA3': 2, 'ACTR1A': 0, 'ADGRA3': 0, 'AKT1S1': 0, 'CDC42EP4': 1, 'CENPB': 10, 'COL6A2': 4, 'CSNK1G2': 0, 'CTCF': 367, 'FAAP100': 3, 'FAM171A1': 0, 'FAM32A': 1, 'FOXD1': 6, 'FZD7': 17, 'GNA11': 8, 'GPR27': 0, 'ICMT': 4, 'KDM4A': 17, 'KIAA2013': 0, 'KLF9': 7, 'MARCKS': 33, 'MEX3D': 0, 'MKI67': 14, 'MLLT1': 3, 'MYOM2': 0, 'NIBAN2': 0, 'NLGN2': 2, 'NORAD': 13, 'NRN1': 1, 'NTN1': 8, 'NXN': 1, 'PLXNA1': 2, 'POM121': 3, 'POM121C': 0, 'PPP1R26': 0, 'PPP1R9B': 1, 'PRMT6': 10, 'PTGFRN': 0, 'PXDN': 2, 'QSOX2': 0, 'RMI2': 5, 'RNF126': 4, 'RRAGA': 1, 'SAPCD2': 0, 'SLC35E2B': 0, 'SMIM13': 0, 'SON': 89, 'SPEN': 18, 'SPEN-AS1': 0, 'TBC1D10B': 0, 'TMEM248': 0, 'TNFRSF1A': 12, 'TOB2': 2, 'WDR6': 1}


In [143]:
very_hiconf_df["hit"] = very_hiconf_df["gene_id"].apply(lambda x: count_dict[x])

In [144]:
very_hiconf_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density,hit
0,NM_001089:5961,ABCA3,2.685794,2.127319,0.655290,0.711017,1,293,296,GGACG,False,0.6536,280.0,0.952834,0.961819,NM_001089,4,6602.0,0.000606,2
8,NM_005736:1364,ACTR1A,2.895120,2.557538,0.803279,0.825807,1,488,493,GGACG,False,0.7918,490.0,0.977891,0.979900,NM_005736,2,2830.0,0.000707,0
15,NM_145290:3525,ADGRA3,2.676393,2.183380,0.695652,0.773499,-1,276,277,GGACG,False,0.6926,270.0,0.956869,0.960855,NM_145290,16,4577.0,0.003496,0
31,NM_032375:532,AKT1S1,3.016171,2.831370,0.890000,0.839323,1,100,101,GGACG,False,0.8750,96.0,0.814711,0.963261,NM_032375,3,2447.0,0.001226,0
119,NM_012121:1462,CDC42EP4,2.722056,2.492623,0.847926,0.826430,1,217,219,GGACG,False,0.8311,219.0,0.915679,0.951848,NM_012121,13,3098.0,0.004196,1
130,NM_001810:825,CENPB,3.157752,2.861978,0.842424,0.913360,-1,330,334,GGACG,False,0.8274,336.0,0.810287,0.982966,NM_001810,14,2890.0,0.004844,10
131,NM_001810:597,CENPB,2.901590,2.442747,0.739394,0.858509,1,330,335,GGACG,False,0.7407,324.0,0.810287,0.982966,NM_001810,14,2890.0,0.004844,10
160,NM_001849:3036,COL6A2,2.526744,2.022392,0.666667,0.760047,-1,492,508,GGACG,False,0.6875,496.0,0.859025,0.978112,NM_001849,2,3445.0,0.000581,4
169,NM_001319:1927,CSNK1G2,3.073604,2.681436,0.793151,0.893732,1,730,748,GGACG,False,0.7699,730.0,0.883239,0.986715,NM_001319,7,2895.0,0.002418,0
173,NM_006565:596,CTCF,2.630316,2.236568,0.745665,0.807812,1,173,173,GGACG,False,0.7314,175.0,0.895219,0.955317,NM_006565,8,3814.0,0.002098,367


In [145]:
gene_list = dorado_fail_df["gene_id"].unique()
count_dict = {gene: get_pubmed_count(gene) for gene in tqdm.tqdm(gene_list)}
dorado_fail_df["hit"] = dorado_fail_df["gene_id"].apply(lambda x: count_dict[x])
dorado_fail_df


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:08<00:00,  1.31it/s]
/tmp/ipykernel_75550/2078951856.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dorado_fail_df["hit"] = dorado_fail_df["gene_id"].apply(lambda x: count_dict[x])


,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density,hit
125,NM_001410823:1309,CDV3,1.869049,1.671504,0.809524,0.678599,1,21,21,GGATT,False,0.0,0.0,0.747327,0.923029,NM_001410823,6,3361.0,0.001785,0
146,NR_171547:2308,CHPF2,2.214009,1.435382,0.380952,0.394277,1,21,21,GGACG,False,0.0,0.0,0.748113,0.921084,NR_171547,13,4206.0,0.003091,0
228,NM_001329090:805,EPHA2,1.608960,1.455806,0.818182,0.941250,-1,22,24,GCACT,False,0.0,0.0,0.872480,0.901347,NM_001329090,10,3878.0,0.002579,49
322,NM_001136557:2193,GPR107,1.787405,1.127730,0.333333,0.214130,1,24,24,GGACG,False,0.0,0.0,0.855416,0.908063,NM_001136557,4,6873.0,0.000582,0
387,NM_152405:308,JMY,1.766405,1.105077,0.333333,0.406751,-1,21,22,GGACG,False,0.0,0.0,0.797200,0.906464,NM_152405,14,9096.0,0.001539,4
390,NM_003724:2200,JRK,1.594822,1.524985,0.909091,0.764723,1,22,25,CAACT,False,0.0,0.0,0.633641,0.903382,NM_003724,15,9097.0,0.001649,2
393,NM_006766:777,KAT6A,1.793477,1.532469,0.750000,0.858974,-1,20,20,GAACG,False,0.0,0.0,0.810287,0.906464,NM_006766,27,9153.0,0.002950,9
408,NR_073498:1533,KHDRBS1,1.532830,0.994549,0.360000,0.545374,1,25,26,GGACG,False,0.0,0.0,0.920141,0.984606,NR_073498,1,2796.0,0.000358,2
488,NM_006077:1871,MICU1,1.987081,1.232901,0.318182,0.261607,1,22,22,GGATT,False,0.0,0.0,0.945733,0.916782,NM_006077,3,2363.0,0.001270,43
552,NM_001278209:2701,NUP153,1.899402,1.336646,0.480000,0.231919,1,25,25,ATACT,False,0.0,0.0,0.906337,0.935234,NM_001278209,15,6119.0,0.002451,21


In [148]:
neg_df = select_df[(select_df["log_pm6a"] == 0.0) & (select_df["dom"] == 0.0) & ((select_df["label"] == 0 )|( select_df["label"] == -3)) & (~select_df["drach"]) & (select_df["cell_perc"] > 0.9)].copy().sort_values("gene_id")
print(neg_df)

                label_id gene_id  log_pm6a  noisy_or  dom  dom_label  label  \
10894242   NM_015665:557    AAAS       0.0      -0.0  0.0   0.000000     -3   
10894038  NM_015665:1283    AAAS       0.0      -0.0  0.0   0.000000     -3   
10894053  NM_015665:1324    AAAS       0.0      -0.0  0.0   0.000000     -3   
10894065  NM_015665:1390    AAAS       0.0      -0.0  0.0   0.000000     -3   
10894068  NM_015665:1395    AAAS       0.0      -0.0  0.0   0.043657     -3   
...                  ...     ...       ...       ...  ...        ...    ...   
10561478  NM_015113:1893   ZZEF1       0.0      -0.0  0.0   0.000000      0   
10561480   NM_015113:190   ZZEF1       0.0      -0.0  0.0   0.000000      0   
10561481  NM_015113:1904   ZZEF1       0.0      -0.0  0.0   0.000000      0   
10561472  NM_015113:1883   ZZEF1       0.0      -0.0  0.0   0.000000      0   
10561974  NM_015113:3828   ZZEF1       0.0      -0.0  0.0   0.000000      0   

          count_dom  count_all   5mer  drach  pred_

In [149]:
from Bio import SeqIO
fasta = "/extdata4/baeklab/Hyeonseo/m6A/res/ref/isoform/hg38_rna_nrnm.fasta"
seq_dict = SeqIO.to_dict(SeqIO.parse(fasta, "fasta"))
seq_dict = {key.split(".")[0]: str(value.seq) for key, value in seq_dict.items()}

In [197]:
refseq_list = [
               "NM_006565",
               "NM_138927",
               "NM_138927",
               "NM_002417",
               "NR_027451",
               "NM_018137",
               "NM_001329090",
               "NM_006766",
               "NM_006077",
               "NM_001278209",
               "NM_001346092",
               "NR_073498",
               ]

pos_list = [
            596,
            3267,
            3243,
            980,
            446,
            1048,
            805,
            777,
            1871,
            2701,
            1994,
            1533]

In [209]:
import RNA
final_df_list = []
for refseq, pos in tqdm.tqdm(zip(refseq_list, pos_list), total=len(refseq_list)):
    refseq_df = neg_df[neg_df["refseq"] == refseq].copy()
    refseq_df["pos"] = refseq_df["label_id"].str.split(":").str[1].astype(int)
    seq = seq_dict[refseq].seq
    refseq_df = refseq_df[(refseq_df["pos"]>50) & (refseq_df["pos"]<(len(seq)-51))]
    refseq_df = refseq_df[(refseq_df["pos"]>pos-200) & (refseq_df["pos"]<(pos+201))].copy()
    refseq_df["seq"] = refseq_df["pos"].apply(lambda x: str(seq[x-50:x+51]))
    refseq_df["mfe"] = refseq_df["seq"].apply(lambda x: RNA.fold(x)[1])
    final_df_list.append(refseq_df)
    if len(refseq_df) == 0:
        raise ValueError("No data")
final_df = pd.concat(final_df_list)
final_df


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:03<00:00,  3.37it/s]


,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,pos,seq,mfe
9345625,NM_006565:403,CTCF,0.0,-0.0,0.0,0.0,0,160,166,AGAGA,False,0.0000,174.0,0.895219,0.955317,NM_006565,403,TGGAGGAGTCCGAAACTTTTATTAAAGGAAAGGAGAGAAAGACTTA...,-22.200001
9345698,NM_006565:635,CTCF,0.0,-0.0,0.0,0.0,0,174,176,GGAGG,False,0.0114,176.0,0.895219,0.955317,NM_006565,635,GGCTGCTGTGGACGATACCCAGATTATAACTTTACAGGTTGTAAAT...,-20.600000
9345708,NM_006565:657,CTCF,0.0,-0.0,0.0,0.0,0,168,174,GGAGA,False,0.0234,171.0,0.895219,0.955317,NM_006565,657,ATTATAACTTTACAGGTTGTAAATATGGAGGAACAGCCCATAAACA...,-22.299999
9345729,NM_006565:741,CTCF,0.0,-0.0,0.0,0.0,0,174,175,GAAAA,False,0.0000,178.0,0.895219,0.955317,NM_006565,741,ACTGTACCTGTTGCTACCACTTCAGTAGAAGAACTTCAGGGGGCTT...,-18.600000
9345732,NM_006565:746,CTCF,0.0,-0.0,0.0,0.0,0,176,177,TGAAG,False,0.0061,163.0,0.895219,0.955317,NM_006565,746,ACCTGTTGCTACCACTTCAGTAGAAGAACTTCAGGGGGCTTATGAA...,-18.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15518876,NR_073498:1720,KHDRBS1,0.0,-0.0,0.0,0.0,0,43,44,GTATG,False,0.0000,22.0,0.920141,0.984606,NR_073498,1720,CTTCTTCTCCCCACCTTATTCCATTCTTAACTCTGCATTCTGGCTT...,-11.700000
15518875,NR_073498:1706,KHDRBS1,0.0,-0.0,0.0,0.0,0,44,44,GCATT,False,0.0000,22.0,0.920141,0.984606,NR_073498,1706,TTCTTCGTGGTCCCCTTCTTCTCCCCACCTTATTCCATTCTTAACT...,-12.100000
15518878,NR_073498:1727,KHDRBS1,0.0,-0.0,0.0,0.0,0,44,44,GTATT,False,0.0000,22.0,0.920141,0.984606,NR_073498,1727,TCCCCACCTTATTCCATTCTTAACTCTGCATTCTGGCTTCTGTATG...,-11.700000
15518879,NR_073498:1732,KHDRBS1,0.0,-0.0,0.0,0.0,0,44,44,TTAAA,False,0.0000,24.0,0.920141,0.984606,NR_073498,1732,ACCTTATTCCATTCTTAACTCTGCATTCTGGCTTCTGTATGTAGTA...,-12.900000


In [224]:
very_hiconf_df = hiconf_df[(hiconf_df["log_pm6a"] > np.percentile(select_df["log_pm6a"],99.8)) & (hiconf_df["cell_perc"] > 0.92)].copy()

exclude_gene = [
    "MKI67",
    "NTN1",
    "RMI2",
    "MICU1",
    "KAT6A",
    "TNFRSF1A",
    "KHDRBS1",
    "CDV3",
    "PRMT6",
    "KDM4A",
    "CENPB",
    "PXDN",
                ]
very_hiconf_df = very_hiconf_df[~very_hiconf_df["gene_id"].isin(exclude_gene)]
very_hiconf_df


,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density
5,NM_001089:5961,ABCA3,2.685794,2.127319,0.655290,0.711017,1,293,296,GGACG,False,0.6536,280.0,0.952834,0.961819,NM_001089,4.0,6602.0,0.000606
27,NM_005736:1364,ACTR1A,2.895120,2.557538,0.803279,0.825807,1,488,493,GGACG,False,0.7918,490.0,0.977891,0.979900,NM_005736,2.0,2830.0,0.000707
37,NM_001116:4959,ADCY9,3.524732,3.454622,0.960000,0.957745,1,150,150,GGACG,False,0.9252,147.0,0.830193,0.932333,NM_001116,21.0,7980.0,0.002632
38,NM_001116:4608,ADCY9,3.266177,3.231568,0.973684,0.962620,1,152,153,GGACG,False,0.9536,151.0,0.830193,0.932333,NM_001116,21.0,7980.0,0.002632
44,NM_145290:3525,ADGRA3,2.676393,2.183380,0.695652,0.773499,-1,276,277,GGACG,False,0.6926,270.0,0.956869,0.960855,NM_145290,16.0,4577.0,0.003496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2893,NM_001171135:2355,ZBED1,2.640257,2.618904,0.973684,0.000000,-3,38,40,GGACG,False,0.9706,34.0,0.820890,0.945790,NM_001171135,30.0,4653.0,0.006447
2897,NM_181842:922,ZBTB12,2.861479,2.715035,0.902985,0.957345,1,134,136,GGAGG,False,0.8397,131.0,0.000000,0.922773,NM_181842,6.0,1963.0,0.003057
2918,NM_015168:3833,ZC3H4,2.721331,2.046289,0.590000,0.752688,1,100,104,GGACG,False,0.5816,98.0,0.859025,0.944513,NM_015168,27.0,6143.0,0.004395
2958,NM_003575:2234,ZNF282,3.129509,2.669289,0.761905,0.718320,1,273,276,GGACG,False,0.7547,265.0,0.731175,0.933231,NM_003575,10.0,3651.0,0.002739


In [225]:

data_df = very_hiconf_df.copy().reset_index(drop=True)
data_df["pos"] = data_df["label_id"].str.split(":").str[1].astype(int)
## filter out pos within 100 bp from the edge
data_df = data_df[(data_df["pos"]>100) & (data_df["pos"]<data_df["refseq"].apply(lambda x: len(seq_dict[x])-100))].copy()

up_seq= "tagccagtaccgtagtgcgtg".upper()
down_seq = "cagaggctgagtcgctgcat".upper()

data_df["up_probe_30"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+31]), axis=1)
data_df["down_probe_30"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-30:x["pos"]])+down_seq, axis=1)
data_df["up_probe_25"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+26]), axis=1)
data_df["down_probe_25"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-25:x["pos"]])+down_seq, axis=1)
data_df["up_probe_20"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+21]), axis=1)
data_df["down_probe_20"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-20:x["pos"]])+down_seq, axis=1)

from Bio.SeqUtils import MeltingTemp as mt
tmfunc = (lambda x: mt.Tm_NN(x, saltcorr=6, nn_table=mt.DNA_NN4))
data_df["tm_up_30"] = data_df["up_probe_30"].apply(tmfunc)
data_df["tm_down_30"] = data_df["down_probe_30"].apply(tmfunc)
data_df["tm_up_25"] = data_df["up_probe_25"].apply(tmfunc)
data_df["tm_down_25"] = data_df["down_probe_25"].apply(tmfunc)
data_df["tm_up_20"] = data_df["up_probe_20"].apply(tmfunc)
data_df["tm_down_20"] = data_df["down_probe_20"].apply(tmfunc)


data_df["tm_max"] = data_df[["tm_up_30", "tm_down_30", "tm_up_25", "tm_down_25", "tm_up_20", "tm_down_20"]].max(axis=1)

data_df = data_df[data_df["tm_max"] < 80].copy()
data_df.drop(columns=["noisy_or", "tm_up_30", "tm_down_30", "tm_up_25", "tm_down_25", "tm_up_20", "tm_down_20",
                      "up_probe_25", "down_probe_25", "up_probe_20", "down_probe_20"
                      ], inplace=True)

repeats = ["A"*5, "C"*5, "G"*5, "T"*5]
data_df["seq"] = data_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-30:x["pos"]+31], axis=1)
data_df["has_repeat"] = data_df["seq"].apply(lambda x: any([repeat in x for repeat in repeats]))
data_df= data_df[~data_df["has_repeat"]]
data_df ["gene_id"].value_counts()
very_hiconf_df = data_df.copy()

very_hiconf_df



,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,refseq,m6a_count,transcript_length,m6a_density,pos,up_probe_30,down_probe_30,tm_max,seq,has_repeat
5,NM_014913:3321,ADNP2,2.610246,0.716867,0.639166,-1,166,170,GGACG,False,...,NM_014913,32.0,5157.0,0.006205,3321,TAGCCAGTACCGTAGTGCGTGGGATCTAATGCTAAAATCTGAAGAG...,CCTTGACAATAGGTCCCTCTGTTCTGCTTTCAGAGGCTGAGTCGCT...,72.690258,AAAGCAGAACAGAGGGACCTATTGTCAAGGACGAGGCTCTTCAGAT...,False
8,NM_032375:481,AKT1S1,2.408016,0.666667,0.650000,-1,93,96,GGACG,False,...,NM_032375,3.0,2447.0,0.001226,481,TAGCCAGTACCGTAGTGCGTGCCACAGTAAACGAAGGCAAGCGACC...,CCCTGACCCACAATAAACTTCTTCGACAAACAGAGGCTGAGTCGCT...,76.384300,TTTGTCGAAGAAGTTTATTGTGGGTCAGGGACGTCAGGTCGCTTGC...,False
14,NM_013275:2171,ANKRD11,2.668826,0.674419,0.748822,1,43,46,GGACG,False,...,NM_013275,27.0,9301.0,0.002903,2171,TAGCCAGTACCGTAGTGCGTGAGGAGTAGTCAGACTCGCTTGTCAG...,CCTTGTGGAGTCTGATAAAGAACTGACCTCCAGAGGCTGAGTCGCT...,73.494963,GAGGTCAGTTCTTTATCAGACTCCACAAGGACGAGACTGACAAGCG...,False
15,NM_001256183:2168,ANKRD11,2.641805,0.696970,0.748822,1,33,38,GGACG,False,...,NM_001256183,29.0,9298.0,0.003119,2168,TAGCCAGTACCGTAGTGCGTGAGGAGTAGTCAGACTCGCTTGTCAG...,CCTTGTGGAGTCTGATAAAGAACTGACCTCCAGAGGCTGAGTCGCT...,73.494963,GAGGTCAGTTCTTTATCAGACTCCACAAGGACGAGACTGACAAGCG...,False
16,NM_001006634:3184,ARHGAP17,3.434832,0.936508,0.894110,1,126,129,GGACG,False,...,NM_001006634,5.0,3495.0,0.001431,3184,TAGCCAGTACCGTAGTGCGTGAGCCTCCCTCCTAATACATAAGAAT...,CCAGGTAGCAGAGAGTAGGCGACTTGCATACAGAGGCTGAGTCGCT...,74.771050,TATGCAAGTCGCCTACTCTCTGCTACCTGGACGTTCATTCTTATGT...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,NM_030775:1378,WNT5B,3.130428,0.939130,0.925638,1,115,118,AGACG,False,...,NM_030775,15.0,2184.0,0.006868,1378,TAGCCAGTACCGTAGTGCGTGGCAAAGTCCACTCCTCAGAGATCTC...,CTCTTAAATAAGCTCTTCCTTTCCATTTTCCAGAGGCTGAGTCGCT...,75.835004,GAAAATGGAAAGGAAGAGCTTATTTAAGAGACGCTGGAGATCTCTG...,False
188,NM_032642:1431,WNT5B,3.130807,0.958904,0.925638,1,146,148,AGACG,False,...,NM_032642,15.0,2238.0,0.006702,1431,TAGCCAGTACCGTAGTGCGTGGCAAAGTCCACTCCTCAGAGATCTC...,CTCTTAAATAAGCTCTTCCTTTCCATTTTCCAGAGGCTGAGTCGCT...,75.835004,GAAAATGGAAAGGAAGAGCTTATTTAAGAGACGCTGGAGATCTCTG...,False
189,NM_016258:1442,YTHDF2,2.736059,0.733711,0.768524,1,353,355,GGACG,False,...,NM_016258,17.0,2744.0,0.006195,1442,TAGCCAGTACCGTAGTGCGTGATATTATACTTAATGGAACGGTGAA...,CCTCAGAGTAGCTCTTAATGATGAAAACCCCAGAGGCTGAGTCGCT...,71.607184,GGGTTTTCATCATTAAGAGCTACTCTGAGGACGATATTCACCGTTC...,False
191,NM_001173128:1521,YTHDF2,2.713925,0.732759,0.768524,1,116,119,GGACG,False,...,NM_001173128,17.0,2823.0,0.006022,1521,TAGCCAGTACCGTAGTGCGTGATATTATACTTAATGGAACGGTGAA...,CCTCAGAGTAGCTCTTAATGATGAAAACCCCAGAGGCTGAGTCGCT...,71.607184,GGGTTTTCATCATTAAGAGCTACTCTGAGGACGATATTCACCGTTC...,False


In [226]:
very_hiconf_df["pos"] = very_hiconf_df["label_id"].str.split(":").str[1].astype(int)

## SET PATH for module import
sys.path.append("/extdata4/baeklab/Hyeonseo/m6A/modformer")
from utils.utils import revcomp_DNA

very_hiconf_df["up_probe"] = very_hiconf_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+31], axis=1)
very_hiconf_df["down_probe"] = very_hiconf_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-30:x["pos"]], axis=1)

In [227]:
up_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(very_hiconf_df["label_id"], very_hiconf_df["up_probe"])])
print(up_probe_fasta)

>NM_014913:3321
CGAGGCTCTTCAGATTTTAGCATTAGATCC
>NM_032375:481
CGTCAGGTCGCTTGCCTTCGTTTACTGTGG
>NM_013275:2171
CGAGACTGACAAGCGAGTCTGACTACTCCT
>NM_001256183:2168
CGAGACTGACAAGCGAGTCTGACTACTCCT
>NM_001006634:3184
CGTTCATTCTTATGTATTAGGAGGGAGGCT
>NM_004491:2100
CGCTAAGATTGAGCACTTGATTAGTTCTCG
>NM_024585:1771
CGCAGTGAGTGTTGCTCAAGGGAGTCAGGA
>NM_032047:1365
CGATGTGTTCATGGGCCTCTGTGCCAATAA
>NM_005180:1768
CGTTAATTGAAAAGAAAGATTGTTGTTATA
>NM_138793:2815
CGTTTCCATCATTCACAGTGCCCTCCCCAC
>NM_015005:2554
CGCTGTGTTATCTAGGAAACCGCTTGCGGC
>NM_004854:1260
CGATGCCCCATACATCTTAAAAGAGGCTGG
>NM_018714:1651
CGTTTCTCCCACACAGGCCAAGAGTTCTGC
>NM_006565:596
CGATACCCAGATTATAACTTTACAGGTTGT
>NM_032656:3836
CGCTGTTCTGTTCAGTGTGCTCTTTGGACT
>NM_014908:1815
TTTTGGGGTCCATCAGCACTGTGTCCCTCC
>NR_033338:3250
TTTCTGGGGTTTGGACTTGGGGTGAGTTTG
>NM_001010924:1590
CGCTGGGGAAAGACTACCATAAGTCAGTGG
>NM_020223:2709
CGTGTACACAGATGCCAATCACCTACCAAA
>NM_003468:1682
CGCTTCCGCTATCCTGAGCGCCCCATCATC
>NM_002048:1328
CGAATGCCGCACCGTCATTGAGGACATGCT
>NM_00

In [228]:
down_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(very_hiconf_df["label_id"], very_hiconf_df["down_probe"])])
print(down_probe_fasta)

>NM_014913:3321
AAAGCAGAACAGAGGGACCTATTGTCAAGG
>NM_032375:481
TTTGTCGAAGAAGTTTATTGTGGGTCAGGG
>NM_013275:2171
GAGGTCAGTTCTTTATCAGACTCCACAAGG
>NM_001256183:2168
GAGGTCAGTTCTTTATCAGACTCCACAAGG
>NM_001006634:3184
TATGCAAGTCGCCTACTCTCTGCTACCTGG
>NM_004491:2100
AGACATGCCCCAGCTGCCCAGCTTGTGTGG
>NM_024585:1771
CCAGAAGTGTATCCCACGAGGGCATCAGGG
>NM_032047:1365
CACAGACACTAAATTCAAGTCTTTACATAG
>NM_005180:1768
AAAGATGGACTACATGTGATACTCCTATGG
>NM_138793:2815
CAGGTTAGGCACGACCCAGGCTGAGAAGGG
>NM_015005:2554
CCTCTTTCTTCATTGGGGACCAGAATGGGG
>NM_004854:1260
GTGTGATTGGACACCACGAGACCCTGGAGG
>NM_018714:1651
TCCCCTCTGATGACTCATCACTGCCCAAGG
>NM_006565:596
CAGTGGCTCCAGAAGCAGAGGCTGCTGTGG
>NM_032656:3836
GGTGGATGATGATTTCATCTCACGTGCTGG
>NM_014908:1815
GGAGTGGACCTAAACTACAGTTATGCTTGG
>NR_033338:3250
ATTCTGGACCCGGAGGGCCAGAGAAACAGG
>NM_001010924:1590
ATCTCCTTTGATAACCTCACTCCAAGTGGG
>NM_020223:2709
TATATTTGATGAATAAGTATATAAACAGAG
>NM_003468:1682
CACAGTGGCCACCTTCCTCATCGACATGGA
>NM_002048:1328
GCAAAGTCTTCAACGGGCTGCGCTGCACGG
>NM_00

In [214]:
very_hiconf_df = hiconf_df[(hiconf_df["log_pm6a"] > np.percentile(select_df["log_pm6a"],99.8)) & (hiconf_df["cell_perc"] > 0.90)].copy()
very_hiconf_df

exclude_gene = [
    "MKI67",
    "NTN1",
    "RMI2",
    "MICU1",
    "KAT6A",
    "TNFRSF1A",
    "KHDRBS1",
    "CDV3",
    "PRMT6",
    "KDM4A",
    "CENPB",
    "PXDN",
]
very_hiconf_df = very_hiconf_df[~very_hiconf_df["gene_id"].isin(exclude_gene)]
very_hiconf_df


,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density
5,NM_001089:5961,ABCA3,2.685794,2.127319,0.655290,0.711017,1,293,296,GGACG,False,0.6536,280.0,0.952834,0.961819,NM_001089,4.0,6602.0,0.000606
14,NM_198147:462,ABHD15,2.393320,2.227509,0.872642,0.942997,-1,212,229,GGACG,False,0.8122,197.0,0.810287,0.919411,NM_198147,29.0,3492.0,0.008305
16,NM_198147:1838,ABHD15,2.998586,2.858266,0.912698,0.950749,-1,252,258,AGACG,False,0.9102,245.0,0.810287,0.919411,NM_198147,29.0,3492.0,0.008305
27,NM_005736:1364,ACTR1A,2.895120,2.557538,0.803279,0.825807,1,488,493,GGACG,False,0.7918,490.0,0.977891,0.979900,NM_005736,2.0,2830.0,0.000707
32,NM_138422:1052,ADAT3,2.389407,1.779976,0.569767,0.586346,-1,86,93,GGACG,False,0.6180,89.0,0.779622,0.913205,NM_138422,14.0,1599.0,0.008755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2958,NM_003575:2234,ZNF282,3.129509,2.669289,0.761905,0.718320,1,273,276,GGACG,False,0.7547,265.0,0.731175,0.933231,NM_003575,10.0,3651.0,0.002739
2991,NM_001195605:938,ZNF865,3.145893,2.856116,0.847222,0.897226,-1,72,76,GGACG,False,0.7639,72.0,0.576541,0.906464,NM_001195605,29.0,3763.0,0.007707
2993,NM_001195605:1756,ZNF865,3.206811,3.156591,0.965517,0.949771,1,87,90,AGACG,False,0.9512,82.0,0.576541,0.906464,NM_001195605,29.0,3763.0,0.007707
2995,NM_023926:2063,ZSCAN18,2.756144,2.249168,0.701754,0.778869,-1,57,63,GGACG,False,0.7083,48.0,0.817941,0.948312,NM_023926,3.0,2783.0,0.001078


KeyError: 'up_count'

In [231]:
blast_columns = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

up_blast_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/blast_up_hits_211024.csv",names=blast_columns)
down_blast_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/blast_down_hits_211024.csv", names=blast_columns)

up_blast_df["qstrand"] = up_blast_df["qstart"] < up_blast_df["qend"]
up_blast_df["sstrand"] = up_blast_df["sstart"] < up_blast_df["send"]
up_blast_df = up_blast_df[up_blast_df["qstrand"] & up_blast_df["sstrand"]]

down_blast_df["qstrand"] = down_blast_df["qstart"] < down_blast_df["qend"]
down_blast_df["sstrand"] = down_blast_df["sstart"] < down_blast_df["send"]
down_blast_df = down_blast_df[down_blast_df["qstrand"] & down_blast_df["sstrand"]]

up_blast_df = up_blast_df[(up_blast_df["length"] >= 19)]
down_blast_df = down_blast_df[(down_blast_df["length"] >= 19)]
up_blast_df = up_blast_df[~((up_blast_df["length"] == 30) & (up_blast_df["pident"] == 100))]
down_blast_df = down_blast_df[~((down_blast_df["length"] == 30) & (down_blast_df["pident"] == 100))]

up_count = up_blast_df.groupby("qseqid").count()["sseqid"]
up_count = up_count.reset_index()
up_count = up_count.rename(columns={"sseqid": "up_count", "qseqid": "label_id"})
down_count = down_blast_df.groupby("qseqid").count()["sseqid"]
down_count = down_count.reset_index()
down_count = down_count.rename(columns={"sseqid": "down_count", "qseqid": "label_id"})

very_hiconf_df = very_hiconf_df.merge(up_count, on="label_id", how="left").merge(down_count, on="label_id", how="left")
very_hiconf_df.fillna(0, inplace=True)
high_conf_blast_df = very_hiconf_df[(very_hiconf_df["up_count"] == 0) & (very_hiconf_df["down_count"] == 0)]
high_conf_blast_df


,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,pos,up_probe_30,down_probe_30,tm_max,seq,has_repeat,up_probe,down_probe,up_count,down_count
0,NM_014913:3321,ADNP2,2.610246,0.716867,0.639166,-1,166,170,GGACG,False,...,3321,TAGCCAGTACCGTAGTGCGTGGGATCTAATGCTAAAATCTGAAGAG...,CCTTGACAATAGGTCCCTCTGTTCTGCTTTCAGAGGCTGAGTCGCT...,72.690258,AAAGCAGAACAGAGGGACCTATTGTCAAGGACGAGGCTCTTCAGAT...,False,CGAGGCTCTTCAGATTTTAGCATTAGATCC,AAAGCAGAACAGAGGGACCTATTGTCAAGG,0.0,0.0
1,NM_032375:481,AKT1S1,2.408016,0.666667,0.650000,-1,93,96,GGACG,False,...,481,TAGCCAGTACCGTAGTGCGTGCCACAGTAAACGAAGGCAAGCGACC...,CCCTGACCCACAATAAACTTCTTCGACAAACAGAGGCTGAGTCGCT...,76.384300,TTTGTCGAAGAAGTTTATTGTGGGTCAGGGACGTCAGGTCGCTTGC...,False,CGTCAGGTCGCTTGCCTTCGTTTACTGTGG,TTTGTCGAAGAAGTTTATTGTGGGTCAGGG,0.0,0.0
5,NM_004491:2100,ARHGAP35,3.293500,0.804878,0.821303,1,82,84,GGACG,False,...,2100,TAGCCAGTACCGTAGTGCGTGCGAGAACTAATCAAGTGCTCAATCT...,CCACACAAGCTGGGCAGCTGGGGCATGTCTCAGAGGCTGAGTCGCT...,79.813633,AGACATGCCCCAGCTGCCCAGCTTGTGTGGACGCTAAGATTGAGCA...,False,CGCTAAGATTGAGCACTTGATTAGTTCTCG,AGACATGCCCCAGCTGCCCAGCTTGTGTGG,0.0,0.0
6,NM_024585:1771,ARMC7,2.801206,0.790541,0.768443,-1,148,149,GGACG,False,...,1771,TAGCCAGTACCGTAGTGCGTGTCCTGACTCCCTTGAGCAACACTCA...,CCCTGATGCCCTCGTGGGATACACTTCTGGCAGAGGCTGAGTCGCT...,78.116857,CCAGAAGTGTATCCCACGAGGGCATCAGGGACGCAGTGAGTGTTGC...,False,CGCAGTGAGTGTTGCTCAAGGGAGTCAGGA,CCAGAAGTGTATCCCACGAGGGCATCAGGG,0.0,0.0
8,NM_005180:1768,BMI1,2.545544,0.925816,0.895357,1,337,339,GGACG,False,...,1768,TAGCCAGTACCGTAGTGCGTGTATAACAACAATCTTTCTTTTCAAT...,CCATAGGAGTATCACATGTAGTCCATCTTTCAGAGGCTGAGTCGCT...,70.512505,AAAGATGGACTACATGTGATACTCCTATGGACGTTAATTGAAAAGA...,False,CGTTAATTGAAAAGAAAGATTGTTGTTATA,AAAGATGGACTACATGTGATACTCCTATGG,0.0,0.0
10,NM_015005:2554,CEP170B,2.972945,0.943662,0.955357,-1,71,74,GGACG,False,...,2554,TAGCCAGTACCGTAGTGCGTGGCCGCAAGCGGTTTCCTAGATAACA...,CCCCATTCTGGTCCCCAATGAAGAAAGAGGCAGAGGCTGAGTCGCT...,76.951973,CCTCTTTCTTCATTGGGGACCAGAATGGGGACGCTGTGTTATCTAG...,False,CGCTGTGTTATCTAGGAAACCGCTTGCGGC,CCTCTTTCTTCATTGGGGACCAGAATGGGG,0.0,0.0
11,NM_004854:1260,CHST10,2.654569,0.611111,0.684800,-1,198,201,GGACG,False,...,1260,TAGCCAGTACCGTAGTGCGTGCCAGCCTCTTTTAAGATGTATGGGG...,CCTCCAGGGTCTCGTGGTGTCCAATCACACCAGAGGCTGAGTCGCT...,77.744087,GTGTGATTGGACACCACGAGACCCTGGAGGACGATGCCCCATACAT...,False,CGATGCCCCATACATCTTAAAAGAGGCTGG,GTGTGATTGGACACCACGAGACCCTGGAGG,0.0,0.0
14,NM_032656:3836,DHX37,2.392832,0.581633,0.559930,1,196,202,GGACG,False,...,3836,TAGCCAGTACCGTAGTGCGTGAGTCCAAAGAGCACACTGAACAGAA...,CCAGCACGTGAGATGAAATCATCATCCACCCAGAGGCTGAGTCGCT...,73.755037,GGTGGATGATGATTTCATCTCACGTGCTGGACGCTGTTCTGTTCAG...,False,CGCTGTTCTGTTCAGTGTGCTCTTTGGACT,GGTGGATGATGATTTCATCTCACGTGCTGG,0.0,0.0
18,NM_020223:2709,FAM20C,2.897343,0.927273,0.938492,-1,275,283,AGACG,False,...,2709,TAGCCAGTACCGTAGTGCGTGTTTGGTAGGTGATTGGCATCTGTGT...,CTCTGTTTATATACTTATTCATCAAATATACAGAGGCTGAGTCGCT...,73.105580,TATATTTGATGAATAAGTATATAAACAGAGACGTGTACACAGATGC...,False,CGTGTACACAGATGCCAATCACCTACCAAA,TATATTTGATGAATAAGTATATAAACAGAG,0.0,0.0
20,NM_002048:1328,GAS1,2.446113,0.545098,0.639591,1,255,263,GGACG,False,...,1328,TAGCCAGTACCGTAGTGCGTGAGCATGTCCTCAATGACGGTGCGGC...,CCGTGCAGCGCAGCCCGTTGAAGACTTTGCCAGAGGCTGAGTCGCT...,79.177090,GCAAAGTCTTCAACGGGCTGCGCTGCACGGACGAATGCCGCACCGT...,False,CGAATGCCGCACCGTCATTGAGGACATGCT,GCAAAGTCTTCAACGGGCTGCGCTGCACGG,0.0,0.0


In [181]:
dorado_fail_df["pos"] = dorado_fail_df["label_id"].str.split(":").str[1].astype(int)

dorado_fail_df = dorado_fail_df[~dorado_fail_df["gene_id"].isin(exclude_gene)]

dorado_fail_df["up_probe"] = dorado_fail_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+31], axis=1)
dorado_fail_df["down_probe"] = dorado_fail_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-30:x["pos"]], axis=1)

dorado_fail_up_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(dorado_fail_df["label_id"], dorado_fail_df["up_probe"])])
dorado_fail_down_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(dorado_fail_df["label_id"], dorado_fail_df["down_probe"])])

In [182]:
print(dorado_fail_up_probe_fasta)

>NM_001013620:638
ACTTCAGCCTTCCTTGGATTTTGTGGCTTC
>NM_020690:2983
ACTTCAGAAAGTATCAGGTAATCAGCAGAT
>NM_018171:3043
TCCCTCCTCCCTATCAAACTAGTATAATGA
>NM_001377448:4028
CGAGGGGGAGCAGCCGGCCCCTGAGGAGGA
>NM_001170629:536
ACTTTCCAAAGAATCCACAGCTCCAGCTCC
>NR_171547:2308
CGCTGCCTCATTGACTCTCTGGGCGTCGGC
>NM_005730:2720
TCATCCCTTCCATTGCCACACCCTAGATCC
>NM_080738:2202
AAAACTACCCAGTGGTGTCTACTGAAGATC
>NM_001329090:805
CTGTGCAGTGGATGGCGAGTGGCTGGTGCC
>NM_001316938:1630
CGGATGAAGAAGGGGTGGGGTTAAGACTCA
>NM_001371462:2217
CGTTGCTTCATCAGGCGTTGCCTCCGGTGT
>NM_001371462:2561
TATTTCTATTCAGATTTATTTATGCCTCTT
>NM_001371462:2250
GTTTGGGGCTTTAGGAAAGCCTGGGTTGGG
>NM_001282618:1722
ACTGCAAATCTAGAAAACTTTTTAGAGAAA
>NM_001136557:2193
CGGCTGGTGGAGGGGGAAGGAGGGTGCGAG
>NM_006041:687
CGCCCCCCAGGCTGCCGTTCCGGGCGCCGC
>NM_001542:603
ACTCCCTCCTTCTGATCTCTTCTTTTTAAT
>NM_152405:308
CGCTTATTGTCCTTCTCTCGATCCGCGCCA
>NM_003724:2200
CTGATGGCTTCTCTGCCCTGCCCTGGCCAC
>NM_004798:4860
ACTTTTCTAAGAGAAGACAGACAAGTTGAC
>NM_032270:1119
ACTTTTCTTGCAATCATACC

In [183]:
print(dorado_fail_down_probe_fasta)

>NM_001013620:638
ATATTTGATGTGTCTTTATGGAAATCATAA
>NM_020690:2983
ACTTAGGTTCTAATGGGACAAATTCTCTTG
>NM_018171:3043
CCAATTCCTGGAACCCCAGCCCACTCCCCC
>NM_001377448:4028
AGCGAGCAGGGAGTCCGGGTGCCCTTGAGG
>NM_001170629:536
TCCCTCCACCAGAGGAAACAGCTCCCACAG
>NR_171547:2308
TCTCAGTGCCCGTCCTGACGAGTGGCTTGG
>NM_005730:2720
AGGGACACCACCCACTCAGGACTCTTCCCC
>NM_080738:2202
TGTCTGGCTGACCTGTACAAGTCCTTCATC
>NM_001329090:805
TGCCACCGGGGGGTGAAGAGCCCCGTATGC
>NM_001316938:1630
GTCACCTTCATGGTCGTCTTCAGGAACAGG
>NM_001371462:2217
TGGAAGGTTAAGGTGGATGCTGTGGTGGGA
>NM_001371462:2561
TGTACAGAAGGATAAAACCCAGGAAAATGG
>NM_001371462:2250
TTGCTTCATCAGGCGTTGCCTCCGGTGTGG
>NM_001282618:1722
TAAAAAAATGAAAGTAAAGGAAAAAAAAAA
>NM_001136557:2193
TGACCCCATGTGTGGGGAAGTGTAGCAAGG
>NM_006041:687
CCGCCAGCCCTGGCCACAGCTCCGGACGGG
>NM_001542:603
ACACTTGCCTCCACTCCTCCCCTTCCCCCC
>NM_152405:308
AGGTGACCATGTGAACTACCTGCTCCCGGG
>NM_003724:2200
CCTTGCCCTGCTCATCCACAGCAGGTGACA
>NM_004798:4860
TCTGACCAAGGGTGGTTAAGTGACACATAG
>NM_032270:1119
GTGGACATTCAGGACATGAC

In [229]:
blast_columns = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

up_blast_df =   pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/blast_up_hits_doradofail_211024.csv",names=blast_columns)
down_blast_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/blast_down_hits_doradofail_211024.csv", names=blast_columns)

up_blast_df["qstrand"] = up_blast_df["qstart"] < up_blast_df["qend"]
up_blast_df["sstrand"] = up_blast_df["sstart"] < up_blast_df["send"]
up_blast_df = up_blast_df[up_blast_df["qstrand"] & up_blast_df["sstrand"]]

down_blast_df["qstrand"] = down_blast_df["qstart"] < down_blast_df["qend"]
down_blast_df["sstrand"] = down_blast_df["sstart"] < down_blast_df["send"]
down_blast_df = down_blast_df[down_blast_df["qstrand"] & down_blast_df["sstrand"]]

up_blast_df = up_blast_df[(up_blast_df["length"] >= 19)]
down_blast_df = down_blast_df[(down_blast_df["length"] >= 19)]
up_blast_df = up_blast_df[~((up_blast_df["length"] == 30) & (up_blast_df["pident"] == 100))]
down_blast_df = down_blast_df[~((down_blast_df["length"] == 30) & (down_blast_df["pident"] == 100))]

up_count = up_blast_df.groupby("qseqid").count()["sseqid"]
up_count = up_count.reset_index()
up_count = up_count.rename(columns={"sseqid": "up_count", "qseqid": "label_id"})
down_count = down_blast_df.groupby("qseqid").count()["sseqid"]
down_count = down_count.reset_index()
down_count = down_count.rename(columns={"sseqid": "down_count", "qseqid": "label_id"})

In [230]:
dorado_fail_blast_df = dorado_fail_df.merge(up_count, on="label_id", how="left").merge(down_count, on="label_id", how="left")
dorado_fail_blast_df.fillna(0, inplace=True)
dorado_fail_blast_df["blast_hit"] = dorado_fail_blast_df["up_count"] + dorado_fail_blast_df["down_count"]
dorado_fail_blast_df = dorado_fail_blast_df[(dorado_fail_blast_df["blast_hit"] == 0)]
dorado_fail_blast_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,...,refseq,m6a_count,transcript_length,m6a_density,pos,up_probe,down_probe,up_count,down_count,blast_hit
4,NM_001170629:536,CHD8,1.165228,1.009193,0.746835,0.000000,0,79,84,AGAAC,...,NM_001170629,20.0,8467.0,0.002362,536,ACTTTCCAAAGAATCCACAGCTCCAGCTCC,TCCCTCCACCAGAGGAAACAGCTCCCACAG,0.0,0.0,0.0
9,NM_001316938:1630,FBXL12,1.174176,0.702675,0.250000,0.410789,1,20,22,GGACG,...,NM_001316938,6.0,1917.0,0.003130,1630,CGGATGAAGAAGGGGTGGGGTTAAGACTCA,GTCACCTTCATGGTCGTCTTCAGGAACAGG,0.0,0.0,0.0
12,NM_001371462:2250,GALNT11,0.947466,0.554661,0.208333,0.630171,-1,24,24,GGAGT,...,NM_001371462,1.0,2861.0,0.000350,2250,GTTTGGGGCTTTAGGAAAGCCTGGGTTGGG,TTGCTTCATCAGGCGTTGCCTCCGGTGTGG,0.0,0.0,0.0
21,NR_040103:1355,NAXD,0.880428,0.808055,0.821429,0.000000,0,28,34,TGAAC,...,NR_040103,10.0,2612.0,0.003828,1355,ACTTTCCTTAGCTCCTTGGTAGTAACTGGG,GTTATGGCGACACTAAACAAAGTATTCCTG,0.0,0.0,0.0
24,NM_001172437:509,PEG10,0.808276,0.450523,0.148148,0.000000,-3,27,28,GGACG,...,NM_001172437,24.0,6639.0,0.003615,509,CGAGCTCTCTGAAGAGATCAACAACTTAAG,GTGTCCCCAACATGACCGAACGAAGAAGGG,0.0,0.0,0.0
26,NM_003913:416,PRPF4B,0.733431,0.521564,0.489510,0.000000,-1,143,166,AGAAC,...,NM_003913,7.0,7517.0,0.000931,416,ACTAAACTTGATGATTTAGCTTTGCTAGAA,TGATAAAGAGGGTATGTCTCCAGCAAAAAG,0.0,0.0,0.0
34,NM_001113361:2293,TBC1D14,0.775750,0.756598,0.931035,0.000000,0,29,33,AAAAC,...,NM_001113361,7.0,4861.0,0.001440,2293,ACTTTTAAAGAATTAAACCAAGGCTTAGCC,GGAGTTGGCCTTAAACAAAACAAACACAAA,0.0,0.0,0.0
36,NM_018052:1605,VAC14,0.893817,0.646648,0.503704,0.000000,-1,135,151,AGACG,...,NM_018052,2.0,3096.0,0.000646,1605,CGTTATCGGATGAATCGGATGAGGTGATCC,ACGGACAGCCTCTTTCCCATCCTACTGCAG,0.0,0.0,0.0
37,NR_001564:546,XIST,0.778385,0.621211,0.609756,0.000000,-3,82,92,TGAAC,...,NR_001564,17.0,19296.0,0.000881,546,ACCCCCAACACTCTGGCCCATCGGGGTGAC,TCGGATACCTGCTGATTCCCTTCCCCTCTG,0.0,0.0,0.0
38,NM_001354729:1390,XPC,1.221978,0.698606,0.192308,0.213089,-1,26,26,GGATT,...,NM_001354729,6.0,3632.0,0.001652,1390,TTCCGAACCTGGCCCTCCAAAGCAGAGGAA,GTGGAGAAGCCTCTGATCCCTCTGATGAGG,0.0,0.0,0.0


In [279]:
get_pubmed_count("CDV3")

0

In [298]:
refseq_list = [
    "NM_001065",
    "NM_001810",
    "NM_012293",
    "NM_012405",
]

pos_list = [
    1876,
    825,
    3195,
    992,
]


In [301]:
import RNA
final_df_list = []
for refseq, pos in tqdm.tqdm(zip(refseq_list, pos_list), total=len(refseq_list)):
    refseq_df = neg_df[neg_df["refseq"] == refseq].copy()
    refseq_df["pos"] = refseq_df["label_id"].str.split(":").str[1].astype(int)
    seq = seq_dict[refseq]
    refseq_df = refseq_df[(refseq_df["pos"]>50) & (refseq_df["pos"]<(len(seq)-51))]
    refseq_df = refseq_df[(refseq_df["pos"]>pos-500) & (refseq_df["pos"]<(pos+501))].copy()
    refseq_df["seq"] = refseq_df["pos"].apply(lambda x: seq[x-50:x+51])
    refseq_df["mfe"] = refseq_df["seq"].apply(lambda x: RNA.fold(x)[1])
    final_df_list.append(refseq_df)
    if len(refseq_df) == 0:
        raise ValueError("No data")
final_df = pd.concat(final_df_list)
final_df

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  5.13it/s]


,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,pos,seq,mfe
965323,NM_001065:1845,TNFRSF1A,0.0,-0.0,0.0,0.0,0,353,353,AGAGC,False,0.0029,349.0,0.890068,0.954823,NM_001065,1845,GCCGACAGTCAGCGCTGTGCGCGCGGAGAGAGGTGCGCCGTGGGCT...,-39.700001
965298,NM_001065:1722,TNFRSF1A,0.0,-0.0,0.0,0.0,0,341,344,CAAGC,False,0.0031,321.0,0.890068,0.954823,NM_001065,1722,TCGCCTTCCAACCCCACTTTTTTCTGGAAAGGAGGGGTCCTGCAGG...,-30.299999
965294,NM_001065:1701,TNFRSF1A,0.0,-0.0,0.0,0.0,0,337,344,AAAGG,False,0.0184,326.0,0.890068,0.954823,NM_001065,1701,TCTAAGGACCGTCCTGCGAGATCGCCTTCCAACCCCACTTTTTTCT...,-31.799999
7246808,NM_001810:968,CENPB,0.0,-0.0,0.0,0.0,0,345,348,AGAAG,False,0.0028,353.0,0.810287,0.982966,NM_001810,968,CCACCCAGCGCCTGAGCGTCCTGCTATGCGCCAATGCCGACGGCAG...,-41.599998
7246794,NM_001810:895,CENPB,0.0,-0.0,0.0,0.0,0,342,343,GGAGG,False,0.0000,341.0,0.810287,0.982966,NM_001810,895,ACCAGTCTATGGTACGACTTCCTGCCCGACCAGGCCGCGGGGCTGT...,-37.799999
7246755,NM_001810:585,CENPB,0.0,-0.0,0.0,0.0,0,325,334,GGAGC,False,0.0032,311.0,0.810287,0.982966,NM_001810,585,GCCGGTCAAGGGCATCATCCTCAAGGAGAAGGCGCTGCGCATAGCC...,-34.299999
7246744,NM_001810:543,CENPB,0.0,-0.0,0.0,0.0,0,329,331,CAAGG,False,0.0000,303.0,0.810287,0.982966,NM_001810,543,CTTGCTCATCGCCTGGTTCCAGCAGATCCGCGCCGCCGGCCTGCCG...,-37.500000
7246717,NM_001810:398,CENPB,0.0,-0.0,0.0,0.0,0,315,319,AGAAC,False,0.0063,315.0,0.810287,0.982966,NM_001810,398,TCGCGCGGCGCTTCAACATCCCGCCGTCCACGCTGAGCACGATCCT...,-33.400002
9800727,NM_012293:2947,PXDN,0.0,-0.0,0.0,0.0,0,192,192,CGAGA,False,0.0054,184.0,0.846608,0.961275,NM_012293,2947,GCCGCTGCTCCCCTTCGCCACCGGGCCGCCCACGGAGTGCATGCGG...,-33.299999
9800731,NM_012293:2955,PXDN,0.0,-0.0,0.0,0.0,0,191,193,AGAGC,False,0.0052,191.0,0.846608,0.961275,NM_012293,2955,TCCCCTTCGCCACCGGGCCGCCCACGGAGTGCATGCGGGACGAGAA...,-32.799999


In [302]:
final_df["pos"] = final_df["label_id"].str.split(":").str[1].astype(int)

final_df["up_probe"] = final_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+31], axis=1)
final_df["down_probe"] = final_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-30:x["pos"]], axis=1)

neg_up_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(final_df["label_id"], final_df["up_probe"])])
neg_down_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(final_df["label_id"], final_df["down_probe"])])

In [303]:
print(neg_up_probe_fasta)

>NM_001065:1845
GCCTGAGTGGGTGGTTTGCGAGGATGAGGG
>NM_001065:1722
GCAGGAGCTAGCAGCCGCCTACTTGGTGCT
>NM_001065:1701
GGAGGGGTCCTGCAGGGGCAAGCAGGAGCT
>NM_001810:968
AGCTGCCCCCGCTGGTGGCCGGCAAGTCGG
>NM_001810:895
GGCGACGGACGGCCGCGTCAAGCCACCCAG
>NM_001810:585
GCTGGGCATGGACGACTTCACCGCCTCCAA
>NM_001810:543
GGGCATCATCCTCAAGGAGAAGGCGCTGCG
>NM_001810:398
ACAAGCGCGCCATCCTGGCGTCGGAGCGCA
>NM_012293:2947
GAACGAGAGCCCCATCCCCTGCTTCCTGGC
>NM_012293:2955
GCCCCATCCCCTGCTTCCTGGCCGGGGACC
>NM_012293:3232
TGCTGGCATCTTCAACGCCTTCGCCACCGC
>NM_012293:2818
GCATGAGGCCCGCAGCATCCGCGACCTGGC
>NM_012293:3624
AAAATGAGATTAAAAACCCTGAGATCCGGG
>NM_012293:3625
AAATGAGATTAAAAACCCTGAGATCCGGGA
>NM_012293:3628
TGAGATTAAAAACCCTGAGATCCGGGAGAA
>NM_012293:3631
GATTAAAAACCCTGAGATCCGGGAGAAACT
>NM_012293:3637
AAACCCTGAGATCCGGGAGAAACTGAAAAG
>NM_012293:2811
GCACGGAGCATGAGGCCCGCAGCATCCGCG
>NM_012293:3648
TCCGGGAGAAACTGAAAAGGTTGTATGGCT
>NM_012293:3657
AACTGAAAAGGTTGTATGGCTCGACACTCA


In [304]:
print(neg_down_probe_fasta)

>NM_001065:1845
GCGCGGAGAGAGGTGCGCCGTGGGCTCAAG
>NM_001065:1722
TTTCTGGAAAGGAGGGGTCCTGCAGGGGCA
>NM_001065:1701
ATCGCCTTCCAACCCCACTTTTTTCTGGAA
>NM_001810:968
CTGCTATGCGCCAATGCCGACGGCAGCGAG
>NM_001810:895
CCTGCCCGACCAGGCCGCGGGGCTGTGCGG
>NM_001810:585
TCAAGGAGAAGGCGCTGCGCATAGCCGAGG
>NM_001810:543
AGCAGATCCGCGCCGCCGGCCTGCCGGTCA
>NM_001810:398
CCGCCGTCCACGCTGAGCACGATCCTGAAG
>NM_012293:2947
CCGGGCCGCCCACGGAGTGCATGCGGGACG
>NM_012293:2955
CCCACGGAGTGCATGCGGGACGAGAACGAG
>NM_012293:3232
GAGAGTACCACGGCTACGACCCCGGCATCA
>NM_012293:2818
TAGACGCATCCAACGTGTACGGGAGCACGG
>NM_012293:3624
CTATCGGCGGCACACACGTTCGAGGACCTG
>NM_012293:3625
TATCGGCGGCACACACGTTCGAGGACCTGA
>NM_012293:3628
CGGCGGCACACACGTTCGAGGACCTGAAAA
>NM_012293:3631
CGGCACACACGTTCGAGGACCTGAAAAATG
>NM_012293:3637
ACACGTTCGAGGACCTGAAAAATGAGATTA
>NM_012293:2811
TCCTACATAGACGCATCCAACGTGTACGGG
>NM_012293:3648
GACCTGAAAAATGAGATTAAAAACCCTGAG
>NM_012293:3657
AATGAGATTAAAAACCCTGAGATCCGGGAG


In [305]:
blast_columns = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

up_blast_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/res/blast/negative2_up_hits.csv",names=blast_columns)
down_blast_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/res/blast/negative2_down_hits.csv", names=blast_columns)

up_blast_df["qstrand"] = up_blast_df["qstart"] < up_blast_df["qend"]
up_blast_df["sstrand"] = up_blast_df["sstart"] < up_blast_df["send"]
up_blast_df = up_blast_df[up_blast_df["qstrand"] & up_blast_df["sstrand"]]

down_blast_df["qstrand"] = down_blast_df["qstart"] < down_blast_df["qend"]
down_blast_df["sstrand"] = down_blast_df["sstart"] < down_blast_df["send"]
down_blast_df = down_blast_df[down_blast_df["qstrand"] & down_blast_df["sstrand"]]

up_blast_df = up_blast_df[(up_blast_df["length"] >= 20)]
down_blast_df = down_blast_df[(down_blast_df["length"] >= 20)]
up_blast_df = up_blast_df[~((up_blast_df["length"] == 30) & (up_blast_df["pident"] == 100))]
down_blast_df = down_blast_df[~((down_blast_df["length"] == 30) & (down_blast_df["pident"] == 100))]

up_count = up_blast_df.groupby("qseqid").count()["sseqid"]
up_count = up_count.reset_index()
up_count = up_count.rename(columns={"sseqid": "up_count", "qseqid": "label_id"})
down_count = down_blast_df.groupby("qseqid").count()["sseqid"]
down_count = down_count.reset_index()
down_count = down_count.rename(columns={"sseqid": "down_count", "qseqid": "label_id"})


In [306]:
neg_blast_df = final_df.merge(up_count, on="label_id", how="left").merge(down_count, on="label_id", how="left")
neg_blast_df.fillna(0, inplace=True)
neg_blast_df["blast_hit"] = neg_blast_df["up_count"] + neg_blast_df["down_count"]
neg_blast_df = neg_blast_df[(neg_blast_df["blast_hit"] == 0)]
neg_blast_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,...,cell_perc,refseq,pos,seq,mfe,up_probe,down_probe,up_count,down_count,blast_hit
4,NM_001810:895,CENPB,0.0,-0.0,0.0,0.0,0,342,343,GGAGG,...,0.982966,NM_001810,895,ACCAGTCTATGGTACGACTTCCTGCCCGACCAGGCCGCGGGGCTGT...,-37.799999,GGCGACGGACGGCCGCGTCAAGCCACCCAG,CCTGCCCGACCAGGCCGCGGGGCTGTGCGG,0.0,0.0,0.0
5,NM_001810:585,CENPB,0.0,-0.0,0.0,0.0,0,325,334,GGAGC,...,0.982966,NM_001810,585,GCCGGTCAAGGGCATCATCCTCAAGGAGAAGGCGCTGCGCATAGCC...,-34.299999,GCTGGGCATGGACGACTTCACCGCCTCCAA,TCAAGGAGAAGGCGCTGCGCATAGCCGAGG,0.0,0.0,0.0
10,NM_012293:3232,PXDN,0.0,-0.0,0.0,0.0,0,187,193,CAATG,...,0.961275,NM_012293,3232,GGTGGGCATGAGGACGCTGGGAGAGTACCACGGCTACGACCCCGGC...,-34.799999,TGCTGGCATCTTCAACGCCTTCGCCACCGC,GAGAGTACCACGGCTACGACCCCGGCATCA,0.0,0.0,0.0
18,NM_012293:3648,PXDN,0.0,-0.0,0.0,0.0,0,205,207,AGATC,...,0.961275,NM_012293,3648,CGGCGGCACACACGTTCGAGGACCTGAAAAATGAGATTAAAAACCC...,-15.100000,TCCGGGAGAAACTGAAAAGGTTGTATGGCT,GACCTGAAAAATGAGATTAAAAACCCTGAG,0.0,0.0,0.0
19,NM_012293:3657,PXDN,0.0,-0.0,0.0,0.0,0,208,208,AGAAA,...,0.961275,NM_012293,3657,ACACGTTCGAGGACCTGAAAAATGAGATTAAAAACCCTGAGATCCG...,-15.700000,AACTGAAAAGGTTGTATGGCTCGACACTCA,AATGAGATTAAAAACCCTGAGATCCGGGAG,0.0,0.0,0.0


In [6]:
agree_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/very_high_conf.tsv", sep="\t", index_col=0).reset_index(drop=True)
agree_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density,hit
0,NM_006565:596,CTCF,2.630316,2.236568,0.745665,0.807812,1,173,173,GGACG,False,0.7314,175,0.895219,0.955317,NM_006565,8,3814,0.002098,367
1,NM_005736:1364,ACTR1A,2.895120,2.557538,0.803279,0.825807,1,488,493,GGACG,False,0.7918,490,0.977891,0.979900,NM_005736,2,2830,0.000707,0
2,NM_145290:3525,ADGRA3,2.676393,2.183380,0.695652,0.773499,-1,276,277,GGACG,False,0.6926,270,0.956869,0.960855,NM_145290,16,4577,0.003496,0
3,NM_032375:532,AKT1S1,3.016171,2.831370,0.890000,0.839323,1,100,101,GGACG,False,0.8750,96,0.814711,0.963261,NM_032375,3,2447,0.001226,0
4,NM_138927:3267,SON,2.415122,2.307777,0.910345,0.924502,1,145,154,GAACG,False,0.8561,132,0.880445,0.967464,NM_138927,36,8393,0.004289,89
5,NM_138927:3243,SON,2.401637,2.207457,0.851351,0.873767,1,148,154,GAACG,False,0.8045,133,0.880445,0.967464,NM_138927,36,8393,0.004289,89
6,NM_015001:741,SPEN,2.883851,2.833272,0.956522,0.923825,1,46,46,GGACG,False,0.9767,43,0.989251,0.983089,NM_015001,42,12385,0.003391,18
7,NM_014663:3515,KDM4A,2.755024,2.458995,0.813370,0.820826,1,359,367,GGACG,False,0.8034,356,0.959576,0.969385,NM_014663,3,4486,0.000669,17
8,NM_001319:1927,CSNK1G2,3.073604,2.681436,0.793151,0.893732,1,730,748,GGACG,False,0.7699,730,0.883239,0.986715,NM_001319,7,2895,0.002418,0
9,NM_002417:980,MKI67,2.361893,2.026780,0.760563,0.711384,-1,71,78,GGACG,False,0.6623,77,0.940261,0.954839,NM_002417,34,12716,0.002674,14


In [7]:
disagree_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/dorado_fail.tsv", sep="\t", index_col=0).reset_index(drop=True)
disagree_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,drach,pred_dorado,count_dorado,ivt_perc,cell_perc,refseq,m6a_count,transcript_length,m6a_density,hit
0,NM_001329090:805,EPHA2,1.608960,1.455806,0.818182,0.941250,-1,22,24,GCACT,False,0,0,0.872480,0.901347,NM_001329090,10,3878,0.002579,49
1,NM_003724:2200,JRK,1.594822,1.524985,0.909091,0.764723,1,22,25,CAACT,False,0,0,0.633641,0.903382,NM_003724,15,9097,0.001649,2
2,NM_006766:777,KAT6A,1.793477,1.532469,0.750000,0.858974,-1,20,20,GAACG,False,0,0,0.810287,0.906464,NM_006766,27,9153,0.002950,9
3,NM_152405:308,JMY,1.766405,1.105077,0.333333,0.406751,-1,21,22,GGACG,False,0,0,0.797200,0.906464,NM_152405,14,9096,0.001539,4
4,NM_001136557:2193,GPR107,1.787405,1.127730,0.333333,0.214130,1,24,24,GGACG,False,0,0,0.855416,0.908063,NM_001136557,4,6873,0.000582,0
5,NM_006077:1871,MICU1,1.987081,1.232901,0.318182,0.261607,1,22,22,GGATT,False,0,0,0.945733,0.916782,NM_006077,3,2363,0.001270,43
6,NR_171547:2308,CHPF2,2.214009,1.435382,0.380952,0.394277,1,21,21,GGACG,False,0,0,0.748113,0.921084,NR_171547,13,4206,0.003091,0
7,NM_001410823:1309,CDV3,1.869049,1.671504,0.809524,0.678599,1,21,21,GGATT,False,0,0,0.747327,0.923029,NM_001410823,6,3361,0.001785,0
8,NM_001278209:2701,NUP153,1.899402,1.336646,0.480000,0.231919,1,25,25,ATACT,False,0,0,0.906337,0.935234,NM_001278209,15,6119,0.002451,21
9,NM_001346092:1994,TNFRSF1A,2.074902,1.581490,0.600000,0.651300,-1,20,21,GGACG,False,0,0,0.827070,0.936626,NM_001346092,2,2289,0.000874,12


In [13]:
agree_df["dorado_agree"] = True
disagree_df["dorado_agree"] = False
data_df = pd.concat([agree_df, disagree_df])
data_df["refseq"] = data_df["label_id"].str.split(":").str[0]
data_df["pos"] = data_df["label_id"].str.split(":").str[1].astype(int)

In [9]:
import RNA

data_df["mfe"] = data_df["seq"].apply(lambda x: RNA.fold(x)[1])
data_df

,label_id,gene_id,log_pm6a,noisy_or,dom,dom_label,label,count_dom,count_all,5mer,...,cell_perc,refseq,m6a_count,transcript_length,m6a_density,hit,dorado_agree,pos,seq,mfe
0,NM_006565:596,CTCF,2.630316,2.236568,0.745665,0.807812,1,173,173,GGACG,...,0.955317,NM_006565,8,3814,0.002098,367,True,596,GACTGAAGTAATGGAGGGCACAGTGGCTCCAGAAGCAGAGGCTGCT...,-26.100000
1,NM_005736:1364,ACTR1A,2.895120,2.557538,0.803279,0.825807,1,488,493,GGACG,...,0.979900,NM_005736,2,2830,0.000707,0,True,1364,GTGTGCACATGCGAGTGCCGTGTGGCCCTGGGACCCTGGGCCCAGA...,-38.000000
2,NM_145290:3525,ADGRA3,2.676393,2.183380,0.695652,0.773499,-1,276,277,GGACG,...,0.960855,NM_145290,16,4577,0.003496,0,True,3525,GGAGCTCGTATTCAGTGCAAGTCAACGTCCAGCCCCCCAACTCTAA...,-24.100000
3,NM_032375:532,AKT1S1,3.016171,2.831370,0.890000,0.839323,1,100,101,GGACG,...,0.963261,NM_032375,3,2447,0.001226,0,True,532,CGTCAGGTCGCTTGCCTTCGTTTACTGTGGTCATGATTGAGCATAT...,-29.000000
4,NM_138927:3267,SON,2.415122,2.307777,0.910345,0.924502,1,145,154,GAACG,...,0.967464,NM_138927,36,8393,0.004289,89,True,3267,GAGCGCTCTATGATGTCAGCTTATGAACGCTCCATGATGTCAGCTT...,-24.299999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6,NR_171547:2308,CHPF2,2.214009,1.435382,0.380952,0.394277,1,21,21,GGACG,...,0.921084,NR_171547,13,4206,0.003091,0,False,2308,GATGGCTGCCGAGGAGACATTCTCAGTGCCCGTCCTGACGAGTGGC...,-34.299999
7,NM_001410823:1309,CDV3,1.869049,1.671504,0.809524,0.678599,1,21,21,GGATT,...,0.923029,NM_001410823,6,3361,0.001785,0,False,1309,CCTATTGCACTATGGAAGTTAAAGTGTCACGACTGCTCTATGCATA...,-15.200000
8,NM_001278209:2701,NUP153,1.899402,1.336646,0.480000,0.231919,1,25,25,ATACT,...,0.935234,NM_001278209,15,6119,0.002451,21,False,2701,CAGACAACAAATGCATAGCCTGTCAAGCAGCAAAATTGTCACCCAG...,-13.900000
9,NM_001346092:1994,TNFRSF1A,2.074902,1.581490,0.600000,0.651300,-1,20,21,GGACG,...,0.936626,NM_001346092,2,2289,0.000874,12,False,1994,GGTGCGCCGTGGGCTCAAGAGCCTGAGTGGGTGGTTTGCGAGGATG...,-34.000000


In [10]:
data_df.drop(columns=["seq","refseq","pos"], inplace=True)

In [11]:
data_df.to_csv("/extdata4/baeklab/Hyeonseo/m6A/select/mfe.tsv", sep="\t", index=False)

In [15]:

sys.path.append("/extdata4/baeklab/Hyeonseo/m6A/modformer")
from utils.utils import revcomp_DNA

In [288]:
data_df = pd.concat([high_conf_blast_df, dorado_fail_blast_df])
data_df["blast_hit"] = data_df["up_count"] + data_df["down_count"]
data_df

,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,down_probe_30,tm_max,seq,has_repeat,up_probe,down_probe,up_count,down_count,noisy_or,blast_hit
0,NM_014913:3321,ADNP2,2.610246,0.716867,0.639166,-1,166,170,GGACG,False,...,CCTTGACAATAGGTCCCTCTGTTCTGCTTTCAGAGGCTGAGTCGCT...,72.690258,AAAGCAGAACAGAGGGACCTATTGTCAAGGACGAGGCTCTTCAGAT...,False,CGAGGCTCTTCAGATTTTAGCATTAGATCC,AAAGCAGAACAGAGGGACCTATTGTCAAGG,0.0,0.0,NaN,0.0
1,NM_032375:481,AKT1S1,2.408016,0.666667,0.650000,-1,93,96,GGACG,False,...,CCCTGACCCACAATAAACTTCTTCGACAAACAGAGGCTGAGTCGCT...,76.384300,TTTGTCGAAGAAGTTTATTGTGGGTCAGGGACGTCAGGTCGCTTGC...,False,CGTCAGGTCGCTTGCCTTCGTTTACTGTGG,TTTGTCGAAGAAGTTTATTGTGGGTCAGGG,0.0,0.0,NaN,0.0
5,NM_004491:2100,ARHGAP35,3.293500,0.804878,0.821303,1,82,84,GGACG,False,...,CCACACAAGCTGGGCAGCTGGGGCATGTCTCAGAGGCTGAGTCGCT...,79.813633,AGACATGCCCCAGCTGCCCAGCTTGTGTGGACGCTAAGATTGAGCA...,False,CGCTAAGATTGAGCACTTGATTAGTTCTCG,AGACATGCCCCAGCTGCCCAGCTTGTGTGG,0.0,0.0,NaN,0.0
6,NM_024585:1771,ARMC7,2.801206,0.790541,0.768443,-1,148,149,GGACG,False,...,CCCTGATGCCCTCGTGGGATACACTTCTGGCAGAGGCTGAGTCGCT...,78.116857,CCAGAAGTGTATCCCACGAGGGCATCAGGGACGCAGTGAGTGTTGC...,False,CGCAGTGAGTGTTGCTCAAGGGAGTCAGGA,CCAGAAGTGTATCCCACGAGGGCATCAGGG,0.0,0.0,NaN,0.0
8,NM_005180:1768,BMI1,2.545544,0.925816,0.895357,1,337,339,GGACG,False,...,CCATAGGAGTATCACATGTAGTCCATCTTTCAGAGGCTGAGTCGCT...,70.512505,AAAGATGGACTACATGTGATACTCCTATGGACGTTAATTGAAAAGA...,False,CGTTAATTGAAAAGAAAGATTGTTGTTATA,AAAGATGGACTACATGTGATACTCCTATGG,0.0,0.0,NaN,0.0
10,NM_015005:2554,CEP170B,2.972945,0.943662,0.955357,-1,71,74,GGACG,False,...,CCCCATTCTGGTCCCCAATGAAGAAAGAGGCAGAGGCTGAGTCGCT...,76.951973,CCTCTTTCTTCATTGGGGACCAGAATGGGGACGCTGTGTTATCTAG...,False,CGCTGTGTTATCTAGGAAACCGCTTGCGGC,CCTCTTTCTTCATTGGGGACCAGAATGGGG,0.0,0.0,NaN,0.0
11,NM_004854:1260,CHST10,2.654569,0.611111,0.684800,-1,198,201,GGACG,False,...,CCTCCAGGGTCTCGTGGTGTCCAATCACACCAGAGGCTGAGTCGCT...,77.744087,GTGTGATTGGACACCACGAGACCCTGGAGGACGATGCCCCATACAT...,False,CGATGCCCCATACATCTTAAAAGAGGCTGG,GTGTGATTGGACACCACGAGACCCTGGAGG,0.0,0.0,NaN,0.0
14,NM_032656:3836,DHX37,2.392832,0.581633,0.559930,1,196,202,GGACG,False,...,CCAGCACGTGAGATGAAATCATCATCCACCCAGAGGCTGAGTCGCT...,73.755037,GGTGGATGATGATTTCATCTCACGTGCTGGACGCTGTTCTGTTCAG...,False,CGCTGTTCTGTTCAGTGTGCTCTTTGGACT,GGTGGATGATGATTTCATCTCACGTGCTGG,0.0,0.0,NaN,0.0
18,NM_020223:2709,FAM20C,2.897343,0.927273,0.938492,-1,275,283,AGACG,False,...,CTCTGTTTATATACTTATTCATCAAATATACAGAGGCTGAGTCGCT...,73.105580,TATATTTGATGAATAAGTATATAAACAGAGACGTGTACACAGATGC...,False,CGTGTACACAGATGCCAATCACCTACCAAA,TATATTTGATGAATAAGTATATAAACAGAG,0.0,0.0,NaN,0.0
20,NM_002048:1328,GAS1,2.446113,0.545098,0.639591,1,255,263,GGACG,False,...,CCGTGCAGCGCAGCCCGTTGAAGACTTTGCCAGAGGCTGAGTCGCT...,79.177090,GCAAAGTCTTCAACGGGCTGCGCTGCACGGACGAATGCCGCACCGT...,False,CGAATGCCGCACCGTCATTGAGGACATGCT,GCAAAGTCTTCAACGGGCTGCGCTGCACGG,0.0,0.0,NaN,0.0


In [289]:
up_seq= "tagccagtaccgtagtgcgtg".upper()
down_seq = "cagaggctgagtcgctgcat".upper()
data_df["up_probe_30"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+31]), axis=1)
data_df["down_probe_30"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-30:x["pos"]])+down_seq, axis=1)
data_df["up_probe_25"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+26]), axis=1)
data_df["down_probe_25"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-25:x["pos"]])+down_seq, axis=1)
data_df["up_probe_20"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+21]), axis=1)
data_df["down_probe_20"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-20:x["pos"]])+down_seq, axis=1)

from Bio.SeqUtils import MeltingTemp as mt
tmfunc = (lambda x: mt.Tm_NN(x, saltcorr=6, nn_table=mt.DNA_NN4))
data_df["tm_up_30"] = data_df["up_probe_30"].apply(tmfunc)
data_df["tm_down_30"] = data_df["down_probe_30"].apply(tmfunc)
data_df["tm_up_25"] = data_df["up_probe_25"].apply(tmfunc)
data_df["tm_down_25"] = data_df["down_probe_25"].apply(tmfunc)
data_df["tm_up_20"] = data_df["up_probe_20"].apply(tmfunc)
data_df["tm_down_20"] = data_df["down_probe_20"].apply(tmfunc)

repeats = ["A"*5, "C"*5, "G"*5, "T"*5]
data_df["seq"] = data_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-30:x["pos"]+31], axis=1)
data_df["has_repeat"] = data_df["seq"].apply(lambda x: any([repeat in x for repeat in repeats]))
data_df= data_df[~data_df["has_repeat"]].copy()

data_df["tm_max"] = data_df[["tm_up_30", "tm_down_30", "tm_up_25", "tm_down_25", "tm_up_20", "tm_down_20"]].max(axis=1)
data_df = data_df[data_df["tm_max"] < 78].copy()
data_df.drop(columns=["noisy_or", "tm_up_30", "tm_down_30", "tm_up_25", "tm_down_25", "tm_up_20", "tm_down_20",
                      "up_probe_25", "down_probe_25", "up_probe_20", "down_probe_20", "up_probe", "down_probe",
                      "up_count", "down_count"
                      ], inplace=True)

pos_site_df = data_df.copy()
pos_site_df

,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,m6a_count,transcript_length,m6a_density,pos,up_probe_30,down_probe_30,tm_max,seq,has_repeat,blast_hit
0,NM_014913:3321,ADNP2,2.610246,0.716867,0.639166,-1,166,170,GGACG,False,...,32.0,5157.0,0.006205,3321,TAGCCAGTACCGTAGTGCGTGGGATCTAATGCTAAAATCTGAAGAG...,CCTTGACAATAGGTCCCTCTGTTCTGCTTTCAGAGGCTGAGTCGCT...,72.690258,AAAGCAGAACAGAGGGACCTATTGTCAAGGACGAGGCTCTTCAGAT...,False,0.0
1,NM_032375:481,AKT1S1,2.408016,0.666667,0.650000,-1,93,96,GGACG,False,...,3.0,2447.0,0.001226,481,TAGCCAGTACCGTAGTGCGTGCCACAGTAAACGAAGGCAAGCGACC...,CCCTGACCCACAATAAACTTCTTCGACAAACAGAGGCTGAGTCGCT...,76.384300,TTTGTCGAAGAAGTTTATTGTGGGTCAGGGACGTCAGGTCGCTTGC...,False,0.0
8,NM_005180:1768,BMI1,2.545544,0.925816,0.895357,1,337,339,GGACG,False,...,6.0,3540.0,0.001695,1768,TAGCCAGTACCGTAGTGCGTGTATAACAACAATCTTTCTTTTCAAT...,CCATAGGAGTATCACATGTAGTCCATCTTTCAGAGGCTGAGTCGCT...,70.512505,AAAGATGGACTACATGTGATACTCCTATGGACGTTAATTGAAAAGA...,False,0.0
10,NM_015005:2554,CEP170B,2.972945,0.943662,0.955357,-1,71,74,GGACG,False,...,20.0,6700.0,0.002985,2554,TAGCCAGTACCGTAGTGCGTGGCCGCAAGCGGTTTCCTAGATAACA...,CCCCATTCTGGTCCCCAATGAAGAAAGAGGCAGAGGCTGAGTCGCT...,76.951973,CCTCTTTCTTCATTGGGGACCAGAATGGGGACGCTGTGTTATCTAG...,False,0.0
11,NM_004854:1260,CHST10,2.654569,0.611111,0.684800,-1,198,201,GGACG,False,...,18.0,2854.0,0.006307,1260,TAGCCAGTACCGTAGTGCGTGCCAGCCTCTTTTAAGATGTATGGGG...,CCTCCAGGGTCTCGTGGTGTCCAATCACACCAGAGGCTGAGTCGCT...,77.744087,GTGTGATTGGACACCACGAGACCCTGGAGGACGATGCCCCATACAT...,False,0.0
14,NM_032656:3836,DHX37,2.392832,0.581633,0.559930,1,196,202,GGACG,False,...,3.0,4559.0,0.000658,3836,TAGCCAGTACCGTAGTGCGTGAGTCCAAAGAGCACACTGAACAGAA...,CCAGCACGTGAGATGAAATCATCATCCACCCAGAGGCTGAGTCGCT...,73.755037,GGTGGATGATGATTTCATCTCACGTGCTGGACGCTGTTCTGTTCAG...,False,0.0
18,NM_020223:2709,FAM20C,2.897343,0.927273,0.938492,-1,275,283,AGACG,False,...,16.0,3176.0,0.005038,2709,TAGCCAGTACCGTAGTGCGTGTTTGGTAGGTGATTGGCATCTGTGT...,CTCTGTTTATATACTTATTCATCAAATATACAGAGGCTGAGTCGCT...,73.105580,TATATTTGATGAATAAGTATATAAACAGAGACGTGTACACAGATGC...,False,0.0
30,NM_015353:1241,KCTD2,3.230837,0.990000,0.962119,1,100,100,GGACG,False,...,17.0,3657.0,0.004649,1241,TAGCCAGTACCGTAGTGCGTGAGGGACGGTGTTCCGAAGCAGCTGC...,CCATTGTGGCACACATAGGGTGAAACCATGCAGAGGCTGAGTCGCT...,77.518362,CATGGTTTCACCCTATGTGTGCCACAATGGACGTTAGCAGCTGCTT...,False,0.0
34,NM_014916:6647,LMTK2,2.626910,0.821782,0.838346,-1,101,103,GGACG,False,...,28.0,8974.0,0.003120,6647,TAGCCAGTACCGTAGTGCGTGAGGTCATTTGCTGCGCACTCTCTGG...,CCGCCTGCATTTTGCACTAGGACTCAACTTCAGAGGCTGAGTCGCT...,76.023264,AAGTTGAGTCCTAGTGCAAAATGCAGGCGGACGTTGCCAGAGAGTG...,False,0.0
35,NM_138699:1421,LOC93622,2.395810,0.714286,0.000000,-3,224,226,GGACG,False,...,9.0,1973.0,0.004562,1421,TAGCCAGTACCGTAGTGCGTGCCACACACTGCCACTCTTCAAGACC...,CCACTGGGCCTTCTTAATCTCATCCAAAGTCAGAGGCTGAGTCGCT...,76.486079,ACTTTGGATGAGATTAAGAAGGCCCAGTGGACGGTTGGTCTTGAAG...,False,0.0


In [290]:
pos_site_df["seq"] = pos_site_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-50:x["pos"]+51], axis=1)
pos_site_df["mfe"] = pos_site_df["seq"].apply(lambda x: RNA.fold(x)[1])
pos_site_df["gc"] = pos_site_df["seq"].apply(lambda x: (x.count("G")+x.count("C"))/len(x))
pos_site_df.drop(columns=["seq"], inplace=True)

In [291]:
pos_site_df

,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,transcript_length,m6a_density,pos,up_probe_30,down_probe_30,tm_max,has_repeat,blast_hit,mfe,gc
0,NM_014913:3321,ADNP2,2.610246,0.716867,0.639166,-1,166,170,GGACG,False,...,5157.0,0.006205,3321,TAGCCAGTACCGTAGTGCGTGGGATCTAATGCTAAAATCTGAAGAG...,CCTTGACAATAGGTCCCTCTGTTCTGCTTTCAGAGGCTGAGTCGCT...,72.690258,False,0.0,-17.299999,0.376238
1,NM_032375:481,AKT1S1,2.408016,0.666667,0.650000,-1,93,96,GGACG,False,...,2447.0,0.001226,481,TAGCCAGTACCGTAGTGCGTGCCACAGTAAACGAAGGCAAGCGACC...,CCCTGACCCACAATAAACTTCTTCGACAAACAGAGGCTGAGTCGCT...,76.384300,False,0.0,-25.000000,0.475248
8,NM_005180:1768,BMI1,2.545544,0.925816,0.895357,1,337,339,GGACG,False,...,3540.0,0.001695,1768,TAGCCAGTACCGTAGTGCGTGTATAACAACAATCTTTCTTTTCAAT...,CCATAGGAGTATCACATGTAGTCCATCTTTCAGAGGCTGAGTCGCT...,70.512505,False,0.0,-10.800000,0.297030
10,NM_015005:2554,CEP170B,2.972945,0.943662,0.955357,-1,71,74,GGACG,False,...,6700.0,0.002985,2554,TAGCCAGTACCGTAGTGCGTGGCCGCAAGCGGTTTCCTAGATAACA...,CCCCATTCTGGTCCCCAATGAAGAAAGAGGCAGAGGCTGAGTCGCT...,76.951973,False,0.0,-33.700001,0.633663
11,NM_004854:1260,CHST10,2.654569,0.611111,0.684800,-1,198,201,GGACG,False,...,2854.0,0.006307,1260,TAGCCAGTACCGTAGTGCGTGCCAGCCTCTTTTAAGATGTATGGGG...,CCTCCAGGGTCTCGTGGTGTCCAATCACACCAGAGGCTGAGTCGCT...,77.744087,False,0.0,-26.700001,0.504950
14,NM_032656:3836,DHX37,2.392832,0.581633,0.559930,1,196,202,GGACG,False,...,4559.0,0.000658,3836,TAGCCAGTACCGTAGTGCGTGAGTCCAAAGAGCACACTGAACAGAA...,CCAGCACGTGAGATGAAATCATCATCCACCCAGAGGCTGAGTCGCT...,73.755037,False,0.0,-28.500000,0.435644
18,NM_020223:2709,FAM20C,2.897343,0.927273,0.938492,-1,275,283,AGACG,False,...,3176.0,0.005038,2709,TAGCCAGTACCGTAGTGCGTGTTTGGTAGGTGATTGGCATCTGTGT...,CTCTGTTTATATACTTATTCATCAAATATACAGAGGCTGAGTCGCT...,73.105580,False,0.0,-9.600000,0.346535
30,NM_015353:1241,KCTD2,3.230837,0.990000,0.962119,1,100,100,GGACG,False,...,3657.0,0.004649,1241,TAGCCAGTACCGTAGTGCGTGAGGGACGGTGTTCCGAAGCAGCTGC...,CCATTGTGGCACACATAGGGTGAAACCATGCAGAGGCTGAGTCGCT...,77.518362,False,0.0,-22.700001,0.574257
34,NM_014916:6647,LMTK2,2.626910,0.821782,0.838346,-1,101,103,GGACG,False,...,8974.0,0.003120,6647,TAGCCAGTACCGTAGTGCGTGAGGTCATTTGCTGCGCACTCTCTGG...,CCGCCTGCATTTTGCACTAGGACTCAACTTCAGAGGCTGAGTCGCT...,76.023264,False,0.0,-27.200001,0.554455
35,NM_138699:1421,LOC93622,2.395810,0.714286,0.000000,-3,224,226,GGACG,False,...,1973.0,0.004562,1421,TAGCCAGTACCGTAGTGCGTGCCACACACTGCCACTCTTCAAGACC...,CCACTGGGCCTTCTTAATCTCATCCAAAGTCAGAGGCTGAGTCGCT...,76.486079,False,0.0,-23.299999,0.495050


In [293]:
pos_site_df = pos_site_df[(pos_site_df["gc"] >= 0.4) & (pos_site_df["gc"] <= 0.6)].copy()
pos_site_df

,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,transcript_length,m6a_density,pos,up_probe_30,down_probe_30,tm_max,has_repeat,blast_hit,mfe,gc
1,NM_032375:481,AKT1S1,2.408016,0.666667,0.650000,-1,93,96,GGACG,False,...,2447.0,0.001226,481,TAGCCAGTACCGTAGTGCGTGCCACAGTAAACGAAGGCAAGCGACC...,CCCTGACCCACAATAAACTTCTTCGACAAACAGAGGCTGAGTCGCT...,76.384300,False,0.0,-25.000000,0.475248
11,NM_004854:1260,CHST10,2.654569,0.611111,0.684800,-1,198,201,GGACG,False,...,2854.0,0.006307,1260,TAGCCAGTACCGTAGTGCGTGCCAGCCTCTTTTAAGATGTATGGGG...,CCTCCAGGGTCTCGTGGTGTCCAATCACACCAGAGGCTGAGTCGCT...,77.744087,False,0.0,-26.700001,0.504950
14,NM_032656:3836,DHX37,2.392832,0.581633,0.559930,1,196,202,GGACG,False,...,4559.0,0.000658,3836,TAGCCAGTACCGTAGTGCGTGAGTCCAAAGAGCACACTGAACAGAA...,CCAGCACGTGAGATGAAATCATCATCCACCCAGAGGCTGAGTCGCT...,73.755037,False,0.0,-28.500000,0.435644
30,NM_015353:1241,KCTD2,3.230837,0.990000,0.962119,1,100,100,GGACG,False,...,3657.0,0.004649,1241,TAGCCAGTACCGTAGTGCGTGAGGGACGGTGTTCCGAAGCAGCTGC...,CCATTGTGGCACACATAGGGTGAAACCATGCAGAGGCTGAGTCGCT...,77.518362,False,0.0,-22.700001,0.574257
34,NM_014916:6647,LMTK2,2.626910,0.821782,0.838346,-1,101,103,GGACG,False,...,8974.0,0.003120,6647,TAGCCAGTACCGTAGTGCGTGAGGTCATTTGCTGCGCACTCTCTGG...,CCGCCTGCATTTTGCACTAGGACTCAACTTCAGAGGCTGAGTCGCT...,76.023264,False,0.0,-27.200001,0.554455
35,NM_138699:1421,LOC93622,2.395810,0.714286,0.000000,-3,224,226,GGACG,False,...,1973.0,0.004562,1421,TAGCCAGTACCGTAGTGCGTGCCACACACTGCCACTCTTCAAGACC...,CCACTGGGCCTTCTTAATCTCATCCAAAGTCAGAGGCTGAGTCGCT...,76.486079,False,0.0,-23.299999,0.495050
52,NM_017841:583,SDHAF2,2.866309,0.905325,0.918439,1,169,170,GGACG,False,...,1186.0,0.004216,583,TAGCCAGTACCGTAGTGCGTGGCATCTAAGAAGCCGGAAGCAAGGC...,CCATCATAAGTAGTTACCATCCACAGACTGCAGAGGCTGAGTCGCT...,75.742049,False,0.0,-26.400000,0.564356
59,NR_024279:2534,SPEN-AS1,2.452236,0.691803,0.000000,-3,305,310,AGACG,False,...,2732.0,0.002928,2534,TAGCCAGTACCGTAGTGCGTGTGCTTAATGGCCAGCCAATGAAAAC...,CTCATTCTACTCCGTCTTTCAGCACGTCAACAGAGGCTGAGTCGCT...,74.057431,False,0.0,-25.799999,0.504950
65,NM_012288:1898,TRAM2,3.093027,0.880597,0.961576,1,67,69,GGACG,False,...,7047.0,0.001561,1898,TAGCCAGTACCGTAGTGCGTGCACACCATCCTTCTGGATTGAGTTG...,CCCAAAGCCAAATGCCTTGGCATTTACAACCAGAGGCTGAGTCGCT...,74.608364,False,0.0,-32.500000,0.534653
4,NM_001170629:536,CHD8,1.165228,0.746835,0.000000,0,79,84,AGAAC,False,...,8467.0,0.002362,536,TAGCCAGTACCGTAGTGCGTGGGAGCTGGAGCTGTGGATTCTTTGG...,CTGTGGGAGCTGTTTCCTCTGGTGGAGGGACAGAGGCTGAGTCGCT...,77.785008,False,0.0,-16.500000,0.495050


In [255]:
neg_df = select_df[(select_df["log_pm6a"] < 0.05) & ((select_df["label"] == 0 )|( select_df["label"] == -3)) & (~select_df["drach"]) & (select_df["cell_perc"] > 0.9)].copy().sort_values("gene_id")
print(neg_df)

                label_id gene_id  log_pm6a  noisy_or       dom  dom_label  \
10894245   NM_015665:569    AAAS -0.000000 -0.000000  0.012000        0.0   
10894190   NM_015665:337    AAAS -0.000000 -0.000000  0.020576        0.0   
10894189   NM_015665:332    AAAS  0.010492  0.005417  0.037118        0.0   
10894187   NM_015665:322    AAAS -0.000000 -0.000000  0.002033        0.0   
10894186   NM_015665:321    AAAS -0.000000 -0.000000  0.012295        0.0   
...                  ...     ...       ...       ...       ...        ...   
10561998   NM_015113:391   ZZEF1  0.000000 -0.000000  0.000000        0.0   
10561999  NM_015113:3914   ZZEF1  0.000000 -0.000000  0.000000        0.0   
10562000  NM_015113:3916   ZZEF1  0.000000 -0.000000  0.000000        0.0   
10561992  NM_015113:3885   ZZEF1  0.000000 -0.000000  0.000000        0.0   
10562802  NM_015113:6850   ZZEF1 -0.000000 -0.000000  0.011628        0.0   

          label  count_dom  count_all   5mer  drach  pred_dorado  \
1089424

In [294]:
refseq_list = pos_site_df["refseq"].unique()

neg_df["refseq"] = neg_df["label_id"].str.split(":").str[0]
neg_site_df = neg_df[neg_df["refseq"].isin(refseq_list)].copy()
neg_site_df["gene_id"].value_counts()

gene_id
CHD8        1920
LMTK2       1794
PEG10       1623
TRAM2       1313
DHX37        757
KCTD2        588
XPC          552
VAC14        530
CHST10       525
LOC93622     365
AKT1S1       328
SPEN-AS1     280
SDHAF2       256
FBXL12        82
Name: count, dtype: int64

In [307]:
up_seq= "tagccagtaccgtagtgcgtg".upper()
down_seq = "cagaggctgagtcgctgcat".upper()

data_df = neg_site_df.copy().reset_index(drop=True)
data_df["pos"] = data_df["label_id"].str.split(":").str[1].astype(int)
data_df["up_probe_30"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+31]), axis=1)
data_df["down_probe_30"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-30:x["pos"]])+down_seq, axis=1)
data_df["up_probe_25"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+26]), axis=1)
data_df["down_probe_25"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-25:x["pos"]])+down_seq, axis=1)
data_df["up_probe_20"] = data_df.apply(lambda x: up_seq+revcomp_DNA(seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+21]), axis=1)
data_df["down_probe_20"] = data_df.apply(lambda x: revcomp_DNA(seq_dict[x["refseq"]][x["pos"]-20:x["pos"]])+down_seq, axis=1)

from Bio.SeqUtils import MeltingTemp as mt
tmfunc = (lambda x: mt.Tm_NN(x, saltcorr=6, nn_table=mt.DNA_NN4))
data_df["tm_up_30"] = data_df["up_probe_30"].apply(tmfunc)
data_df["tm_down_30"] = data_df["down_probe_30"].apply(tmfunc)
data_df["tm_up_25"] = data_df["up_probe_25"].apply(tmfunc)
data_df["tm_down_25"] = data_df["down_probe_25"].apply(tmfunc)
data_df["tm_up_20"] = data_df["up_probe_20"].apply(tmfunc)
data_df["tm_down_20"] = data_df["down_probe_20"].apply(tmfunc)

repeats = ["A"*5, "C"*5, "G"*5, "T"*5]
data_df["seq"] = data_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-30:x["pos"]+31], axis=1)
data_df["has_repeat"] = data_df["seq"].apply(lambda x: any([repeat in x for repeat in repeats]))
data_df= data_df[~data_df["has_repeat"]].copy()

data_df["tm_max"] = data_df[["tm_up_30", "tm_down_30", "tm_up_25", "tm_down_25", "tm_up_20", "tm_down_20"]].max(axis=1)
data_df = data_df[data_df["tm_max"] < 75].copy()
data_df.drop(columns=["tm_up_30", "tm_down_30", "tm_up_25", "tm_down_25", "tm_up_20", "tm_down_20",
                      "up_probe_25", "down_probe_25", "up_probe_20", "down_probe_20", "seq"
                      ], inplace=True)

In [308]:
data_df[data_df["log_pm6a"] == 0.0]["gene_id"].value_counts()

gene_id
PEG10       612
CHD8        519
LMTK2       362
TRAM2       242
XPC          86
LOC93622     75
KCTD2        60
CHST10       55
VAC14        46
SDHAF2       41
DHX37        30
SPEN-AS1     20
FBXL12       17
AKT1S1        9
Name: count, dtype: int64

In [309]:
data_df = data_df[data_df["log_pm6a"] == 0.0].copy()

In [310]:
data_df["gene_id"].value_counts()

gene_id
PEG10       612
CHD8        519
LMTK2       362
TRAM2       242
XPC          86
LOC93622     75
KCTD2        60
CHST10       55
VAC14        46
SDHAF2       41
DHX37        30
SPEN-AS1     20
FBXL12       17
AKT1S1        9
Name: count, dtype: int64

In [311]:
data_df["seq"] = data_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-50:x["pos"]+51], axis=1)
data_df = data_df[data_df["seq"].apply(len) == 101].copy()
data_df["mfe"] = data_df["seq"].apply(lambda x: RNA.fold(x)[1])
data_df["gc"] = data_df["seq"].apply(lambda x: (x.count("G")+x.count("C"))/len(x))
data_df.drop(columns=["seq"], inplace=True)

data_df = data_df[(data_df["gc"] >= 0.4) & (data_df["gc"] <= 0.6) & (data_df["mfe"] > -30)].copy()
data_df["gene_id"].value_counts()

,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,ivt_perc,cell_perc,refseq,pos,up_probe_30,down_probe_30,has_repeat,tm_max,mfe,gc
1,NM_032375:459,AKT1S1,-0.0,0.012048,0.0,0,83,91,GAAGA,False,...,0.814711,0.963261,NM_032375,459,TAGCCAGTACCGTAGTGCGTGACCTGACGTCCCTGACCCACAATAA...,TCGACAAAAGTAGAGGAAATCGGTGCTCGTCAGAGGCTGAGTCGCT...,False,74.293441,-25.299999,0.485149
6,NM_032375:1971,AKT1S1,-0.0,0.029412,0.0,0,408,429,CCAGT,False,...,0.814711,0.963261,NM_032375,1971,TAGCCAGTACCGTAGTGCGTGAGGAAAAGCGGACGGTGACTATGTA...,GGTCAAGCCCTTTACAGGATTTGAGCCAATCAGAGGCTGAGTCGCT...,False,73.560597,-23.299999,0.485149
8,NM_032375:1816,AKT1S1,-0.0,0.064171,0.0,0,374,401,TTAAG,False,...,0.814711,0.963261,NM_032375,1816,TAGCCAGTACCGTAGTGCGTGTAGAGCTTAGAACTCAGCGAGCCAA...,AATAGAAGGAATCTGTCGCTAGGCGGAGAGCAGAGGCTGAGTCGCT...,False,74.412695,-33.799999,0.495050
9,NM_001170629:7079,CHD8,-0.0,0.022599,0.0,0,177,181,TGAGG,False,...,0.903009,0.967135,NM_001170629,7079,TAGCCAGTACCGTAGTGCGTGTGCTTCTGGAATGTTAACTTCAATC...,CATCTTTGATTTGAACTGTAAACTCCTTTTCAGAGGCTGAGTCGCT...,False,70.798859,-20.900000,0.425743
10,NM_001170629:7074,CHD8,0.0,0.000000,0.0,0,190,192,AAAGA,False,...,0.903009,0.967135,NM_001170629,7074,TAGCCAGTACCGTAGTGCGTGCTGGAATGTTAACTTCAATCCTTCC...,TTGATTTGAACTGTAAACTCCTTTTCATCCCAGAGGCTGAGTCGCT...,False,70.636140,-20.200001,0.415842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2167,NM_001354729:973,XPC,0.0,0.000000,0.0,0,20,20,ACAGC,False,...,0.900671,0.920128,NM_001354729,973,TAGCCAGTACCGTAGTGCGTGTTTGCTGTTGCTGACTTCAGAGGAA...,GTAGAGACAATACCAGCCGGGTCAAGAGCTCAGAGGCTGAGTCGCT...,False,74.840894,-20.100000,0.514851
2170,NM_001354729:2106,XPC,-0.0,0.033333,0.0,0,30,31,TGAGG,False,...,0.900671,0.920128,NM_001354729,2106,TAGCCAGTACCGTAGTGCGTGTCACCATCTTGTAGGGTACTTCTCC...,CACCACTCTTGCTTTCTTCAGCCACGTGTCCAGAGGCTGAGTCGCT...,False,74.811292,-28.600000,0.504950
2171,NM_001354729:2116,XPC,0.0,0.000000,0.0,0,31,31,AGAAG,False,...,0.900671,0.920128,NM_001354729,2116,TAGCCAGTACCGTAGTGCGTGGAAAAGCCTTTCACCATCTTGTAGG...,CTCCAAGCCTCACCACTCTTGCTTTCTTCACAGAGGCTGAGTCGCT...,False,73.802031,-26.799999,0.524752
2172,NM_001354729:2117,XPC,0.0,0.000000,0.0,0,31,31,GAAGT,False,...,0.900671,0.920128,NM_001354729,2117,TAGCCAGTACCGTAGTGCGTGAGAAAAGCCTTTCACCATCTTGTAG...,TCTCCAAGCCTCACCACTCTTGCTTTCTTCCAGAGGCTGAGTCGCT...,False,74.410102,-24.500000,0.524752


In [312]:
data_df["gene_id"].value_counts()

gene_id
CHD8        444
PEG10       326
TRAM2       183
LMTK2       149
KCTD2        55
LOC93622     49
XPC          47
VAC14        43
CHST10       36
SDHAF2       35
DHX37        28
FBXL12        9
SPEN-AS1      6
AKT1S1        3
Name: count, dtype: int64

In [314]:
data_df = data_df[(data_df["gc"] >= 0.4) & (data_df["gc"] <= 0.6) & (data_df["mfe"] > -30)].copy()

In [315]:
neg_site_df = data_df.copy()

## sample 10 sites per gene
cand_list = []
for gene_id in neg_site_df["gene_id"].unique():
    gene_df = neg_site_df[neg_site_df["gene_id"] == gene_id].copy()
    gene_df = gene_df.sample(min(10, len(gene_df)))
    cand_list.append(gene_df)
neg_site_df = pd.concat(cand_list)

neg_site_df

,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,ivt_perc,cell_perc,refseq,pos,up_probe_30,down_probe_30,has_repeat,tm_max,mfe,gc
6,NM_032375:1971,AKT1S1,-0.0,0.029412,0.0,0,408,429,CCAGT,False,...,0.814711,0.963261,NM_032375,1971,TAGCCAGTACCGTAGTGCGTGAGGAAAAGCGGACGGTGACTATGTA...,GGTCAAGCCCTTTACAGGATTTGAGCCAATCAGAGGCTGAGTCGCT...,False,73.560597,-23.299999,0.485149
1,NM_032375:459,AKT1S1,-0.0,0.012048,0.0,0,83,91,GAAGA,False,...,0.814711,0.963261,NM_032375,459,TAGCCAGTACCGTAGTGCGTGACCTGACGTCCCTGACCCACAATAA...,TCGACAAAAGTAGAGGAAATCGGTGCTCGTCAGAGGCTGAGTCGCT...,False,74.293441,-25.299999,0.485149
334,NM_001170629:3089,CHD8,-0.0,0.017857,0.0,0,112,115,TGAGA,False,...,0.903009,0.967135,NM_001170629,3089,TAGCCAGTACCGTAGTGCGTGCGAAGCTCAGGACAATCTGACAAAA...,CAAAAGTGGTGATCAGAGCGTCAAACTTGTCAGAGGCTGAGTCGCT...,False,71.114928,-22.700001,0.465347
504,NM_001170629:1933,CHD8,0.0,0.000000,0.0,0,97,98,AGAAG,False,...,0.903009,0.967135,NM_001170629,1933,TAGCCAGTACCGTAGTGCGTGAATTATCAGATGAGGTATTACGTTT...,CTTACCCACTACAGGAGTGATGGTGTTGAGCAGAGGCTGAGTCGCT...,False,73.867675,-24.900000,0.425743
327,NM_001170629:3102,CHD8,-0.0,0.008929,0.0,0,112,115,TCAGA,False,...,0.903009,0.967135,NM_001170629,3102,TAGCCAGTACCGTAGTGCGTGCCATTCAATTTCACGAAGCTCAGGA...,GACAAAATCATCTCAAAAGTGGTGATCAGACAGAGGCTGAGTCGCT...,False,71.631519,-18.200001,0.415842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2164,NM_001354729:998,XPC,0.0,0.000000,0.0,0,20,20,ACAGC,False,...,0.900671,0.920128,NM_001354729,998,TAGCCAGTACCGTAGTGCGTGTCTTTCCTTGGAAGGTTTCTTTCCC...,GTTGCTGACTTCAGAGGAATTGGCTGTAGACAGAGGCTGAGTCGCT...,False,73.496918,-22.200001,0.495050
2104,NM_001354729:1106,XPC,0.0,0.000000,0.0,0,20,20,AAAGG,False,...,0.900671,0.920128,NM_001354729,1106,TAGCCAGTACCGTAGTGCGTGCTTAGCAAAGGTTTCCTCTTGTTTG...,TTGCTGGTCTTTGGTTTGGTGTGGTTTTCTCAGAGGCTGAGTCGCT...,False,73.470188,-18.200001,0.485149
2155,NM_001354729:3421,XPC,0.0,0.000000,0.0,0,38,39,CCAAG,False,...,0.900671,0.920128,NM_001354729,3421,TAGCCAGTACCGTAGTGCGTGTTTAGGCTTCTGTCACCACAAAATG...,GGAAAACTAGATCCCAGCAGATGACCTGTACAGAGGCTGAGTCGCT...,False,72.645052,-17.200001,0.415842
2093,NM_001354729:1546,XPC,-0.0,0.038462,0.0,0,26,26,CAAGA,False,...,0.900671,0.920128,NM_001354729,1546,TAGCCAGTACCGTAGTGCGTGTCTGCCTTCTCACCATCGCTGCACA...,TGCCTCTTTTACTGCTTGAAGAGCTTGAGGCAGAGGCTGAGTCGCT...,False,73.806027,-20.400000,0.485149


In [316]:
neg_site_df["pos"] = neg_site_df["label_id"].str.split(":").str[1].astype(int)

neg_site_df["up_probe"] = neg_site_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]+1:x["pos"]+31], axis=1)
neg_site_df["down_probe"] = neg_site_df.apply(lambda x: seq_dict[x["refseq"]][x["pos"]-30:x["pos"]], axis=1)

neg_up_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(neg_site_df["label_id"], neg_site_df["up_probe"])])
neg_down_probe_fasta = "\n".join([f">{label_id}\n{seq}" for label_id, seq in zip(neg_site_df["label_id"], neg_site_df["down_probe"])])

In [318]:
print(neg_up_probe_fasta)

>NM_032375:1971
GTCTCTACATAGTCACCGTCCGCTTTTCCT
>NM_032375:459
GAAGTTTATTGTGGGTCAGGGACGTCAGGT
>NM_001170629:3089
GATGATTTTGTCAGATTGTCCTGAGCTTCG
>NM_001170629:1933
AGAGAAAACGTAATACCTCATCTGATAATT
>NM_001170629:3102
GATTGTCCTGAGCTTCGTGAAATTGAATGG
>NM_001170629:4064
GGTGTACCGCCTCATCACTCGTAATTCCTA
>NM_001170629:2223
GATGCAGCCATTGTAGACAAAGTGCTTTCT
>NM_001170629:3376
TTCTTAAGCCAATGATGCTGAGAAGACTCA
>NM_001170629:7132
TGGGAGATGGACATCCACTGTTTCATAAGA
>NM_001170629:2531
GCCCGTTATTTACTACCTGGTGAAATGGTG
>NM_001170629:3836
ACGTATTGATGGGCGAGTTAGAGGCAACCT
>NM_001170629:3718
TTGACAAGTTGCTTCCAAAGCTTAAAGCTG
>NM_004854:495
TGTGTACAGTGCCAAACAGGAGTTTCTGTT
>NM_004854:1644
GTTTGGATGTAAGATGCTCTGAGGACCCTG
>NM_004854:689
AGAATCTCTCGCACACTCCTGTCTCCAAGT
>NM_004854:486
AGACCCAGATGTGTACAGTGCCAAACAGGA
>NM_004854:736
TTTGTCTGTGACAAGCACAAGATTCTTTTC
>NM_004854:753
CAAGATTCTTTTCTGCCAGACTCCCAAAGT
>NM_004854:2456
TTTCGATTCCTTTCATCTCAGCAAAATGGG
>NM_004854:2039
TTTTGTTCCTGTAGACAGTCAGCGATGGCC
>NM_004854:452
TGGTGGCTAGCAAGTTCATCA

In [319]:
print(neg_down_probe_fasta)

>NM_032375:1971
ATTGGCTCAAATCCTGTAAAGGGCTTGACC
>NM_032375:459
ACGAGCACCGATTTCCTCTACTTTTGTCGA
>NM_001170629:3089
ACAAGTTTGACGCTCTGATCACCACTTTTG
>NM_001170629:1933
CTCAACACCATCACTCCTGTAGTGGGTAAG
>NM_001170629:3102
TCTGATCACCACTTTTGAGATGATTTTGTC
>NM_001170629:4064
GTCATCGAATTGGGCAGAGCAAAGCTGTGA
>NM_001170629:2223
GCAGTTCTTTGTGGAGAATCCCAGTGAAGA
>NM_001170629:3376
ACAGAGGAACAGGTTCAAAAGCTACAGGCC
>NM_001170629:7132
CAGAAGCACAAGTTGATGGCGAATGGAGTA
>NM_001170629:2531
AGTCTCACAGTATTGACAAGGACAATGGGG
>NM_001170629:3836
ATTATTTAATCCAGAGGAGGTACTTATATG
>NM_001170629:3718
ATGGTTCGTTCAGCCGGCAAACTGGTTCTT
>NM_004854:495
AGTTCATCACGTTGACCTTTAAAGACCCAG
>NM_004854:1644
TATGACGCAGAACCCCAACTGTTACAACTT
>NM_004854:689
ATCAGAAACGTCTGCAGGGATGATGCCCTG
>NM_004854:486
TGGCTAGCAAGTTCATCACGTTGACCTTTA
>NM_004854:736
TCCTGTCTCCAAGTTTGTCCTGGACCGAAT
>NM_004854:753
TCCTGGACCGAATATTTGTCTGTGACAAGC
>NM_004854:2456
CAGCATCACAGAATTCAGTGTAGTTTATAC
>NM_004854:2039
TTGTGGAGGCAAAGCATTCTTTCTGTGACT
>NM_004854:452
GCCGCATGCTTTTGGGTGATT

In [321]:
blast_columns = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

up_blast_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/blast_neg_up_hits_211024.csv",names=blast_columns)
down_blast_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/select/blast_neg_down_hits_211024.csv", names=blast_columns)

up_blast_df["qstrand"] = up_blast_df["qstart"] < up_blast_df["qend"]
up_blast_df["sstrand"] = up_blast_df["sstart"] < up_blast_df["send"]
up_blast_df = up_blast_df[up_blast_df["qstrand"] & up_blast_df["sstrand"]]

down_blast_df["qstrand"] = down_blast_df["qstart"] < down_blast_df["qend"]
down_blast_df["sstrand"] = down_blast_df["sstart"] < down_blast_df["send"]
down_blast_df = down_blast_df[down_blast_df["qstrand"] & down_blast_df["sstrand"]]

up_blast_df = up_blast_df[(up_blast_df["length"] >= 20)]
down_blast_df = down_blast_df[(down_blast_df["length"] >= 20)]
up_blast_df = up_blast_df[~((up_blast_df["length"] == 30) & (up_blast_df["pident"] == 100))]
down_blast_df = down_blast_df[~((down_blast_df["length"] == 30) & (down_blast_df["pident"] == 100))]

up_count = up_blast_df.groupby("qseqid").count()["sseqid"]
up_count = up_count.reset_index()
up_count = up_count.rename(columns={"sseqid": "up_count", "qseqid": "label_id"})
down_count = down_blast_df.groupby("qseqid").count()["sseqid"]
down_count = down_count.reset_index()
down_count = down_count.rename(columns={"sseqid": "down_count", "qseqid": "label_id"})

In [322]:
neg_blast_df = neg_site_df.merge(up_count, on="label_id", how="left").merge(down_count, on="label_id", how="left")
neg_blast_df.fillna(0, inplace=True)
neg_blast_df["blast_hit"] = neg_blast_df["up_count"] + neg_blast_df["down_count"]
neg_blast_df = neg_blast_df[(neg_blast_df["blast_hit"] == 0)]
neg_blast_df

,label_id,gene_id,log_pm6a,dom,dom_label,label,count_dom,count_all,5mer,drach,...,down_probe_30,has_repeat,tm_max,mfe,gc,up_probe,down_probe,up_count,down_count,blast_hit
0,NM_032375:1971,AKT1S1,-0.0,0.029412,0.0,0,408,429,CCAGT,False,...,GGTCAAGCCCTTTACAGGATTTGAGCCAATCAGAGGCTGAGTCGCT...,False,73.560597,-23.299999,0.485149,GTCTCTACATAGTCACCGTCCGCTTTTCCT,ATTGGCTCAAATCCTGTAAAGGGCTTGACC,0.0,0.0,0.0
3,NM_001170629:1933,CHD8,0.0,0.000000,0.0,0,97,98,AGAAG,False,...,CTTACCCACTACAGGAGTGATGGTGTTGAGCAGAGGCTGAGTCGCT...,False,73.867675,-24.900000,0.425743,AGAGAAAACGTAATACCTCATCTGATAATT,CTCAACACCATCACTCCTGTAGTGGGTAAG,0.0,0.0,0.0
8,NM_001170629:7132,CHD8,-0.0,0.005464,0.0,0,183,192,TAATG,False,...,TACTCCATTCGCCATCAACTTGTGCTTCTGCAGAGGCTGAGTCGCT...,False,73.923924,-19.299999,0.415842,TGGGAGATGGACATCCACTGTTTCATAAGA,CAGAAGCACAAGTTGATGGCGAATGGAGTA,0.0,0.0,0.0
10,NM_001170629:3836,CHD8,0.0,0.000000,0.0,0,126,129,TGAAC,False,...,CATATAAGTACCTCCTCTGGATTAAATAATCAGAGGCTGAGTCGCT...,False,74.914789,-26.299999,0.465347,ACGTATTGATGGGCGAGTTAGAGGCAACCT,ATTATTTAATCCAGAGGAGGTACTTATATG,0.0,0.0,0.0
11,NM_001170629:3718,CHD8,-0.0,0.007937,0.0,0,126,131,TTATT,False,...,AAGAACCAGTTTGCCGGCTGAACGAACCATCAGAGGCTGAGTCGCT...,False,74.937033,-20.200001,0.465347,TTGACAAGTTGCTTCCAAAGCTTAAAGCTG,ATGGTTCGTTCAGCCGGCAAACTGGTTCTT,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,NM_001354729:1086,XPC,0.0,0.000000,0.0,0,20,20,ACACC,False,...,GTGGTTTTCTAGAACTTGGCTGGAAGTTTCCAGAGGCTGAGTCGCT...,False,74.295792,-19.600000,0.475248,CCAAACCAAAGACCAGCAAAGGAACCAAAC,GAAACTTCCAGCCAAGTTCTAGAAAACCAC,0.0,0.0,0.0
120,NM_001354729:1538,XPC,0.0,0.000000,0.0,0,25,26,AAAAG,False,...,TTACTGCTTGAAGAGCTTGAGGATGCCGCTCAGAGGCTGAGTCGCT...,False,74.621176,-17.500000,0.485149,AGAGGCAAGAAAATGTGCAGCGATGGTGAG,AGCGGCATCCTCAAGCTCTTCAAGCAGTAA,0.0,0.0,0.0
121,NM_001354729:1539,XPC,0.0,0.000000,0.0,0,26,26,AAAGA,False,...,TTTACTGCTTGAAGAGCTTGAGGATGCCGCCAGAGGCTGAGTCGCT...,False,74.701489,-17.500000,0.475248,GAGGCAAGAAAATGTGCAGCGATGGTGAGA,GCGGCATCCTCAAGCTCTTCAAGCAGTAAA,0.0,0.0,0.0
122,NM_001354729:998,XPC,0.0,0.000000,0.0,0,20,20,ACAGC,False,...,GTTGCTGACTTCAGAGGAATTGGCTGTAGACAGAGGCTGAGTCGCT...,False,73.496918,-22.200001,0.495050,GCAAAGGGAAAGAAACCTTCCAAGGAAAGA,TCTACAGCCAATTCCTCTGAAGTCAGCAAC,0.0,0.0,0.0


In [323]:
neg_blast_df["gene_id"].value_counts()

gene_id
SDHAF2      8
LOC93622    8
PEG10       7
LMTK2       7
CHST10      7
TRAM2       7
XPC         7
KCTD2       6
SPEN-AS1    6
VAC14       6
CHD8        4
DHX37       3
FBXL12      2
AKT1S1      1
Name: count, dtype: int64

In [122]:
neg_blast_df.drop(columns=["up_probe_30", "down_probe_30", "seq"], inplace=True)
neg_blast_df.to_csv("/extdata4/baeklab/Hyeonseo/m6A/select/neg_blast.tsv", sep = "\t")

In [324]:
print(pos_site_df)

             label_id   gene_id  log_pm6a       dom  dom_label  label  \
1       NM_032375:481    AKT1S1  2.408016  0.666667   0.650000     -1   
11     NM_004854:1260    CHST10  2.654569  0.611111   0.684800     -1   
14     NM_032656:3836     DHX37  2.392832  0.581633   0.559930      1   
30     NM_015353:1241     KCTD2  3.230837  0.990000   0.962119      1   
34     NM_014916:6647     LMTK2  2.626910  0.821782   0.838346     -1   
35     NM_138699:1421  LOC93622  2.395810  0.714286   0.000000     -3   
52      NM_017841:583    SDHAF2  2.866309  0.905325   0.918439      1   
59     NR_024279:2534  SPEN-AS1  2.452236  0.691803   0.000000     -3   
65     NM_012288:1898     TRAM2  3.093027  0.880597   0.961576      1   
4    NM_001170629:536      CHD8  1.165228  0.746835   0.000000      0   
9   NM_001316938:1630    FBXL12  1.174176  0.250000   0.410789      1   
24   NM_001172437:509     PEG10  0.808276  0.148148   0.000000     -3   
36     NM_018052:1605     VAC14  0.893817  0.503704

In [325]:
print(neg_blast_df)

              label_id gene_id  log_pm6a       dom  dom_label  label  \
0       NM_032375:1971  AKT1S1      -0.0  0.029412        0.0      0   
3    NM_001170629:1933    CHD8       0.0  0.000000        0.0      0   
8    NM_001170629:7132    CHD8      -0.0  0.005464        0.0      0   
10   NM_001170629:3836    CHD8       0.0  0.000000        0.0      0   
11   NM_001170629:3718    CHD8      -0.0  0.007937        0.0      0   
..                 ...     ...       ...       ...        ...    ...   
119  NM_001354729:1086     XPC       0.0  0.000000        0.0      0   
120  NM_001354729:1538     XPC       0.0  0.000000        0.0      0   
121  NM_001354729:1539     XPC       0.0  0.000000        0.0      0   
122   NM_001354729:998     XPC       0.0  0.000000        0.0      0   
124  NM_001354729:3421     XPC       0.0  0.000000        0.0      0   

     count_dom  count_all   5mer  drach  ...  \
0          408        429  CCAGT  False  ...   
3           97         98  AGAAG  False

In [326]:
pos_site_df.to_csv("/extdata4/baeklab/Hyeonseo/m6A/select/pos_site_211024.tsv", sep="\t", index=False)
neg_blast_df.to_csv("/extdata4/baeklab/Hyeonseo/m6A/select/neg_site_211024.tsv", sep="\t", index=False)